# POC segmentation CT cardiaque — exécution dans Colab

Ce notebook exécute les scripts du dépôt GitHub sur le GPU de Colab. Les données vont dans ton Google Drive.

**Avant de commencer** : menu **Exécution → Modifier le type d'exécution → GPU T4**. Puis exécute les cellules une par une (▶).

## 1. Relier Drive et vérifier le GPU
Une fenêtre d'autorisation s'ouvre : accepte avec ton **compte Google personnel**. Tu dois voir **Tesla T4**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Récupérer le dépôt GitHub
Le dépôt GitHub `Cameronprr/poc-segmentation-ct` est copié dans Colab (cette copie disparaît à la fin de la session ; le dépôt sur GitHub reste la référence).

In [ ]:
%cd /content
# On sort d'abord de l'ancien dossier du dépôt : sinon, après sa suppression, le dossier courant
# n'existe plus et le shell affiche l'erreur « getcwd ».
PSEUDO = 'Cameronprr'
!rm -rf /content/poc-segmentation-ct
!git clone https://github.com/{PSEUDO}/poc-segmentation-ct.git /content/poc-segmentation-ct
%cd /content/poc-segmentation-ct
!ls

## 3. Installer les bibliothèques de l'étape
Pour l'étape 2, seule `nibabel` (lecture des fichiers NIfTI) est nécessaire. Les autres (MONAI, nnU-Net, TotalSegmentator) seront installées aux étapes suivantes.

In [ ]:
!pip -q install nibabel

## 4. Étape 2 — Télécharger MM-WHS dans Drive
À ne faire qu'une fois : les données restent ensuite dans ton Drive. `megatools` est un petit outil qui télécharge depuis le service MEGA. Compte plusieurs minutes.

In [ ]:
DONNEES = '/content/drive/MyDrive/poc_ct/mmwhs'
!mkdir -p "$DONNEES"
!apt-get -qq install megatools > /dev/null
!megadl --path "$DONNEES" "https://mega.nz/folder/UNMF2YYI#1cqJVzo4p_wESv9P_pc8uA"

# Décompresser d'éventuelles archives
import glob, zipfile, os
for z in glob.glob(DONNEES + '/**/*.zip', recursive=True):
    print('Décompression :', z)
    zipfile.ZipFile(z).extractall(os.path.dirname(z))

## 5. Étape 2 — Vérifier les données
Le script `scripts/02_verifier_donnees.py` contrôle que les 20 cas du découpage figé sont complets et affiche leurs caractéristiques. **Résultat attendu : CONFORME — 20 cas complets.**

In [ ]:
!python scripts/02_verifier_donnees.py --donnees "$DONNEES"

**Fin de l'étape 2.** Copie le tableau affiché (ou le message d'erreur) et envoie-le dans la conversation.

## 6. Étape 3 — Installer MONAI
MONAI est une bibliothèque d'imagerie médicale (chargement des scanners, transformations, métriques). Le script de l'étape 3 s'en sert pour lire les volumes et calculer les miniatures. À refaire à chaque nouvelle session Colab, comme les cellules 1 et 2.

In [ ]:
!pip -q install "monai>=1.3" nibabel

## 7. Étape 3 — Contrôle qualité détaillé et conversion au format nnU-Net
Le script `scripts/03_preparer.py` contrôle les 20 cas : forme, affine, espacement, orientation, étiquettes et doublons entre test et entraînement. Il écrit ensuite le rapport `rapport_qc_etape3.csv`. Si tout est conforme, il écrit aussi le jeu `Dataset101_MMWHS` dans `poc_ct/nnunet_raw`.

**Résultat attendu :** « Résultat du contrôle : CONFORME », puis 15 cas dans imagesTr/labelsTr et 5 dans imagesTs/labelsTs.

Si le script s'arrête, le dernier message indique pourquoi :
- **NON CONFORME** : rien n'a été écrit au format nnU-Net ;
- **existe déjà** : le jeu a déjà été écrit. Ne relance avec `--ecraser` que si tu veux vraiment le remplacer ;
- **ERREUR PENDANT LA CONVERSION** : le dossier est incomplet. Relance avec `--ecraser` une fois la cause corrigée.

Les lignes « QUASI-DOUBLON possible » ne bloquent pas : envoie-les dans la conversation pour qu'on les examine.

In [ ]:
DONNEES = '/content/drive/MyDrive/poc_ct/mmwhs'
SORTIE = '/content/drive/MyDrive/poc_ct/nnunet_raw'
!python scripts/03_preparer.py --donnees "$DONNEES" --sortie "$SORTIE"

### 7 bis. Reconstruire `Dataset101_MMWHS` avec le nouveau découpage (décision D20)
**Avant :** relance les sections 1, 2 et 6. La section 2 récupère le dépôt à jour, avec le nouveau `splits/mmwhs_split.json`. Sinon, le script reconstruirait l'**ancien** découpage sans rien signaler. Pour vérifier, `!head -12 splits/mmwhs_split.json` doit montrer en test 1004, 1010, 1011, 1013 et 1014.

Le découpage figé a été remplacé une fois (D20) : le jeu nnU-Net déjà écrit dans Drive suit l'**ancien** découpage. Cette variante relance le même script avec `--ecraser` : l'ancien dossier `Dataset101_MMWHS` est **supprimé**, puis réécrit avec le découpage de `splits/mmwhs_split.json`. Le rapport `rapport_qc_etape3.csv` est aussi remplacé.

**À n'exécuter qu'une fois, maintenant.** N'exécute pas la cellule 7 juste au-dessus : elle s'arrêterait sur « existe déjà ». Une exception : si le script affiche **ERREUR PENDANT LA CONVERSION**, le dossier est incomplet. Corrige la cause, puis relance cette même cellule.

**Résultat attendu :**
- « Résultat du contrôle : CONFORME » ;
- en test `mmwhs_1004`, `1010`, `1011`, `1013` et `1014` (5 cas dans imagesTs/labelsTs), et 15 cas dans imagesTr/labelsTr ;
- la liste des paires test/entraînement de corrélation supérieure à 0,95, à m'envoyer pour qu'on vérifie qu'aucune n'est suspecte.

Si le script s'arrête sur une règle D20 (patient à plusieurs examens en test), rien n'a été modifié dans Drive. S'il affiche NON CONFORME, `Dataset101_MMWHS` n'est pas touché, mais `rapport_qc_etape3.csv` est quand même remplacé.

In [ ]:
DONNEES = '/content/drive/MyDrive/poc_ct/mmwhs'
SORTIE = '/content/drive/MyDrive/poc_ct/nnunet_raw'
!python scripts/03_preparer.py --donnees "$DONNEES" --sortie "$SORTIE" --ecraser

**Fin du script 03.** Copie tout ce que le script a affiché (tableau, ressemblances, avertissements, résultat) et envoie-le dans la conversation pour la vérification par l'agent QA.

## 8. Étape 3 — Examiner à l'œil les deux quasi-doublons possibles
**Examen fait le 8 novembre 2026, conclusion dans D20.** Depuis la reconstruction (section 7 bis), 1005 et 1020 sont en `imagesTr` : cette cellule afficherait « Fichier introuvable ». **Ne plus la relancer.**

Le script 03 a signalé deux paires (scanner de test / scanner d'entraînement) qui ont exactement la même taille et le même espacement, avec des miniatures très corrélées :
- `mmwhs_1005` (test) / `mmwhs_1012` (entraînement), corrélation 0,962 ;
- `mmwhs_1020` (test) / `mmwhs_1001` (entraînement), corrélation 0,974.

Si l'une de ces paires était **le même examen** (ou le même patient), le modèle aurait « vu » pendant l'entraînement un cas très proche du lot de test, et le verdict final serait trop optimiste. Mais attention : même taille et même espacement ne prouvent rien (beaucoup de scanners ont ces réglages), et **une corrélation de 0,96-0,97 entre miniatures est attendue entre deux thorax différents cadrés de la même façon**. Ce n'est pas un signal fort en soi : il faut regarder.

**Vocabulaire**
- **Coupe axiale** : une tranche horizontale du corps (comme une tranche de saucisson), vue depuis les pieds du patient. On y voit le cœur, les côtes en arc, le sternum devant et la colonne vertébrale derrière.
- **Coupe coronale** : une tranche verticale, vue de face (le patient debout face à toi). Souvent plus parlante pour reconnaître une silhouette : forme du thorax, du cœur, des poumons, hauteur du diaphragme.
- **Coupe du milieu** : la tranche située à mi-hauteur (axiale) ou à mi-profondeur (coronale) du volume.
- **Sens d'affichage** : celui des radiologues. Sur les deux vues, **la droite du patient est à gauche de l'image** (comme quand on fait face à quelqu'un). L'avant du patient est en haut de la coupe axiale, la tête en haut de la coupe coronale. Les deux scanners d'une paire sont affichés exactement dans le même sens.

**Ce qu'affiche la cellule** : pour chaque paire, une figure de 2 lignes (axiale en haut, coronale en bas) et 3 colonnes : scanner de test, scanner d'entraînement, puis leur **différence** (test − entraînement). Les deux scanners utilisent la même fenêtre d'affichage, de -200 à 800 HU (la même que le script 03).

**Comment lire la carte de différence** (blanc = 0 ; rouge = le test est plus clair ; bleu = l'entraînement est plus clair ; au-delà de ±500 HU la couleur sature) :
- **carte toute blanche** (au plus un léger bruit) : c'est **le même examen**, donc un doublon ;
- **carte non blanche** (contours en paires rouge/bleu) : ce sont des **examens différents**, mais **pas forcément des patients différents** : le même patient scanné un autre jour, ou à une autre phase du battement cardiaque, donnerait aussi des contours. Et un doublon simplement décalé d'une ou deux coupes, ou recadré, donnerait lui aussi des contours rouge/bleu : d'où le « meilleur décalage » ci-dessous.

**Pour juger « même patient »**, compare l'anatomie dans les deux premières colonnes : forme des vertèbres, des côtes et du sternum, calcifications (petites taches très blanches, par exemple sur l'aorte ou les coronaires), forme générale du thorax, taille du cœur.

**Les chiffres affichés sous chaque figure**
- **forme, espacement, orientation, origine** : l'origine est la position dans l'espace (en mm) du premier voxel du volume, lue dans l'en-tête du fichier. Deux examens distincts ont souvent des origines différentes ; une origine identique au centième de mm est un indice de copie (pas une preuve).
- **différence absolue moyenne** (axiale, coronale) : moyenne de |test − entraînement| sur la coupe affichée, en HU. 0,0 veut dire des coupes identiques au voxel près **dans la fenêtre affichée** (-200 à 800 HU). C'est un repère : l'air autour du corps vaut -200 dans les deux images (différence nulle), ce qui fait baisser la moyenne.
- **meilleur décalage** : la coupe axiale du milieu du test est comparée aux 21 coupes de l'entraînement situées de 10 coupes en dessous à 10 coupes au-dessus. On affiche le décalage qui donne la plus petite différence, et cette différence. Un doublon décalé donnerait une différence proche de 0 pour un décalage non nul ; deux examens différents restent loin de 0 quel que soit le décalage.
- **volumes entiers identiques ?** : on calcule l'empreinte (SHA-256, une suite de 64 caractères tirée de tous les voxels) de chaque volume. « IDENTIQUES » = même volume au voxel près (doublon exact) ; « différents » = au moins un voxel diffère. Le script 03 a déjà fait ce contrôle ; il est refait ici pour avoir tous les éléments au même endroit.

**Avant de lancer** : après un redémarrage de session, relance la section 1 (Drive) ; nibabel et matplotlib sont préinstallés dans Colab. La cellule ne fait que **lire** deux scanners par paire, un à la fois : elle n'écrit aucun fichier, ne lit aucun masque (le dossier `labelsTs` reste fermé, décision D18) et ne touche pas au découpage figé.

Envoie ta conclusion dans la conversation ; la décision (garder, ou retirer un cas) sera tracée dans le journal.

In [ ]:
# Comparaison à l'œil des deux paires signalées par le script 03. LECTURE SEULE : rien n'est écrit.
# Seuls les scanners (images) sont lus : aucun masque, et surtout pas labelsTs (décision D18).
import hashlib, os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

DATASET = '/content/drive/MyDrive/poc_ct/nnunet_raw/Dataset101_MMWHS'
# (numéro du cas de test, numéro du cas d'entraînement, corrélation des miniatures donnée par le script 03)
PAIRES = [('1005', '1012', 0.962),
          ('1020', '1001', 0.974)]
FENETRE = (-200, 800)   # fenêtre d'affichage en HU, identique pour les deux scanners (comme le script 03)
ECHELLE_DIFF = 500      # carte de différence de -500 à +500 HU (au-delà, la couleur sature)
DECALAGE_MAX = 10       # recherche du meilleur décalage : coupes k-10 à k+10 de l'entraînement

def vue_radiologique(coupe):
    """Écrête la coupe dans la FENETRE et la met dans le sens des radiologues pour imshow(origin='lower') :
    np.flip sur le 1er axe = droite du patient à gauche de l'image ; .T = avant (ou tête) en haut."""
    return np.flip(np.clip(coupe.astype(np.float32), *FENETRE), 0).T

def lire(chemin, k_ref=None):
    """Lit UN scanner. Renvoie ses coupes du milieu (axiale, coronale), quelques infos, et, si k_ref est
    donné, les coupes axiales k_ref-10..k_ref+10 (pour le meilleur décalage). Le volume complet n'existe
    que dans cette fonction : la mémoire est libérée à la sortie."""
    img = nib.load(chemin)
    # Remise en orientation RAS (1er axe vers la droite du patient, 2e vers l'avant, 3e vers le haut) :
    # ainsi les deux scanners d'une paire sont toujours traités dans le même sens.
    ras = nib.as_closest_canonical(img)
    vol = np.asanyarray(ras.dataobj)                              # voxels en HU
    h = hashlib.sha256(f'{vol.shape}{vol.dtype}'.encode())        # empreinte de TOUT le volume
    h.update(np.ascontiguousarray(vol).tobytes())
    sx, sy, sz = np.sqrt((ras.affine[:3, :3] ** 2).sum(axis=0))   # espacement en mm le long de chaque axe
    infos = {'forme': img.shape,
             'espacement': tuple(round(float(z), 3) for z in img.header.get_zooms()[:3]),
             'orientation': ''.join(nib.aff2axcodes(img.affine)),
             'origine': tuple(round(float(v), 2) for v in img.affine[:3, 3]),   # position du 1er voxel (mm)
             'empreinte': h.hexdigest()}
    k, j = vol.shape[2] // 2, vol.shape[1] // 2   # milieu en hauteur (axiale), milieu avant-arrière (coronale)
    # aspect = rapport des espacements, pour ne pas écraser l'image si les voxels ne sont pas cubiques.
    vues = {'axiale':   (vue_radiologique(vol[:, :, k]), k, vol.shape[2], sy / sx),
            'coronale': (vue_radiologique(vol[:, j, :]), j, vol.shape[1], sz / sx)}
    voisines = {}
    if k_ref is not None:
        for d in range(-DECALAGE_MAX, DECALAGE_MAX + 1):
            if 0 <= k_ref + d < vol.shape[2]:
                voisines[d] = vue_radiologique(vol[:, :, k_ref + d])
    del vol, ras, img
    return vues, infos, voisines

for test, entr, r in PAIRES:
    ch_test = f'{DATASET}/imagesTs/mmwhs_{test}_0000.nii.gz'
    ch_entr = f'{DATASET}/imagesTr/mmwhs_{entr}_0000.nii.gz'
    manquants = [c for c in (ch_test, ch_entr) if not os.path.exists(c)]
    if manquants:
        print('Fichier introuvable (Drive est-il relié ? section 1) :', manquants)
        continue
    v_t, i_t, _ = lire(ch_test)                                   # un volume à la fois
    v_e, i_e, voisines = lire(ch_entr, k_ref=v_t['axiale'][1])
    lus = [('TEST', f'mmwhs_{test}', v_t, i_t), ('entraînement', f'mmwhs_{entr}', v_e, i_e)]

    fig, axes = plt.subplots(2, 3, figsize=(16, 11))
    fig.suptitle(f'mmwhs_{test} (TEST) contre mmwhs_{entr} (entraînement) — corrélation des miniatures {r}'
                 '\n(droite du patient à gauche de l\'image)', fontsize=14)
    ecarts = {}
    for ligne, vue in enumerate(['axiale', 'coronale']):
        for col, (lot, nom, vues, _) in enumerate(lus):
            coupe, num, total, aspect = vues[vue]
            axes[ligne, col].imshow(coupe, cmap='gray', vmin=FENETRE[0], vmax=FENETRE[1],
                                    origin='lower', aspect=aspect)
            axes[ligne, col].set_title(f'{nom} — lot {lot}\ncoupe {vue} n°{num + 1} sur {total}')
        c_t, num_t, total_t, aspect_t = v_t[vue]                  # infos du TEST pour la carte de différence
        c_e, aspect_e = v_e[vue][0], v_e[vue][3]
        ax = axes[ligne, 2]
        if c_t.shape == c_e.shape and np.isclose(aspect_t, aspect_e):
            diff = c_t - c_e
            ecarts[vue] = float(np.abs(diff).mean())
            im = ax.imshow(diff, cmap='RdBu_r', vmin=-ECHELLE_DIFF, vmax=ECHELLE_DIFF,
                           origin='lower', aspect=aspect_t)       # RdBu_r : bleu < 0 < rouge, blanc = 0
            fig.colorbar(im, ax=ax, shrink=0.8, label='test − entraînement (HU)')
            ax.set_title(f'Différence test − entraînement\ncoupe {vue} n°{num_t + 1} sur {total_t}')
        else:
            ax.text(0.5, 0.5, 'Formes ou espacements différents :\ndifférence non calculable',
                    ha='center', va='center')
    for ax in axes.ravel():
        ax.axis('off')   # pas de graduations : seules les images comptent
    plt.tight_layout()
    plt.show()

    # Quelques chiffres simples sous la figure
    for lot, nom, _, infos in lus:
        print(f"{nom:<12} lot {lot:<13} forme {infos['forme']}  espacement {infos['espacement']} mm  "
              f"orientation {infos['orientation']}  origine {infos['origine']} mm")
    for vue, e in ecarts.items():
        print(f'Différence absolue moyenne, coupe {vue} (fenêtre {FENETRE[0]} à {FENETRE[1]} HU) : {e:.1f} HU')
    # Meilleur décalage : coupe axiale du milieu du test contre les coupes voisines de l'entraînement
    ref = v_t['axiale'][0]
    par_decalage = {d: float(np.abs(ref - c).mean()) for d, c in voisines.items() if c.shape == ref.shape}
    if par_decalage:
        d = min(par_decalage, key=par_decalage.get)
        print(f'Meilleur décalage (axiale, de -{DECALAGE_MAX} à +{DECALAGE_MAX} coupes) : {d:+d} coupe(s), '
              f'différence absolue moyenne {par_decalage[d]:.1f} HU')
    else:
        print('Meilleur décalage : non calculable (formes différentes)')
    print('Volumes entiers (empreinte SHA-256) :',
          'IDENTIQUES (doublon exact)' if i_t['empreinte'] == i_e['empreinte'] else 'différents')
    print()
    del lus, v_t, v_e, voisines

## 9. Découpage par patient (décision D20)
**Pourquoi.** L'examen de la section 8 a montré que les paires `1005`/`1012` et `1020`/`1001` ont la même forme, le même espacement et la même origine au centième de mm, mais des voxels différents : très probablement **le même patient scanné à deux phases du battement cardiaque** (cœur contracté ou relâché). Or `1005` et `1020` sont en test, `1012` et `1001` en entraînement. Un même patient des deux côtés fausse le verdict : le modèle aurait déjà « vu » ce patient. On refait donc le découpage **par patient** : le lot de test ne contient que des **patients à un seul examen** (5 cas = 5 patients indépendants, chaque patient compte une fois dans le verdict) ; les patients à plusieurs examens vont, entiers, en entraînement.

**Ce que fait le script `scripts/03b_regrouper_patients.py`.**
1. Il lit uniquement l'**en-tête** des 20 scanners (taille, espacement, position, orientation) : c'est instantané, aucun voxel n'est chargé et aucun masque n'est lu.
2. Il regroupe en « patients probables » les cas qui ont la même forme, le même espacement et la même **affine** (la matrice qui place le volume dans l'espace, origine comprise) à 0,01 mm près, et affiche tous les groupes.
3. Il vérifie que la position est **réellement enregistrée** dans l'en-tête : les colonnes `sform/qform` donnent deux codes qui le disent. S'ils valent tous les deux 0, il n'y a pas de position enregistrée : nibabel en fabrique une par défaut, identique pour tous les volumes de même taille. Le cas est alors marqué **« origine NON FIABLE »** et n'est regroupé avec aucun autre.
4. Il liste pour information les **paires proches** : même plan de coupe et même position gauche-droite et avant-arrière (à 1 mm près), mais un nombre de coupes ou une hauteur différents. Elles ne sont pas regroupées automatiquement.
5. Il tire au sort, avec la graine 42, 5 patients **à un seul examen** (et à origine fiable) pour le lot de test. Le tirage est **stratifié par épaisseur de coupe** (ici l'espacement entre deux coupes, lu dans l'en-tête) : 1 patient à 0,45 mm et 4 à 0,625 mm, pour que le test contienne les deux types d'examens. Tous les autres cas, dont les deux paires, vont en entraînement / validation croisée.
6. Il affiche la proposition, les cas qui changent de lot, et écrit le fichier `mmwhs_split_par_patient_PROPOSITION.json` dans `poc_ct` (Drive). **Le dépôt et `splits/mmwhs_split.json` ne sont pas modifiés** : la proposition y sera recopiée seulement après validation.

La géométrie seule ne prouve rien : elle s'ajoute à l'examen visuel de la section 8.

**Résultat attendu.** Au moins les groupes `{1001, 1020}` et `{1005, 1012}`, signalés « FUITE dans le découpage actuel », puis un lot de test de 5 cas (1 à 0,45 mm, 4 à 0,625 mm) qui ne contient **aucun** de ces quatre cas. Normalement, `sform/qform` n'affiche jamais `0/0`. À examiner ensemble, s'ils apparaissent :
- une ligne « ATTENTION » (groupe de plus de 2 cas, ou paire proche à cheval sur test et entraînement dans la proposition) ;
- le cadre de points d'exclamation **« ORIGINE NON FIABLE »** à la toute fin ;
- un « ARRÊT » : épaisseur de coupe imprévue (ni 0,45 ni 0,625 mm), 3e axe qui n'est pas l'axe des coupes, ou une épaisseur sans assez de patients à un seul examen pour remplir ses places (par exemple aucun à 0,45 mm). Dans ce cas, rien n'est écrit.

**Ensuite.** Copie **toute** la sortie de la cellule, y compris le contenu JSON affiché à la fin, et envoie-la dans la conversation. La reconversion au format nnU-Net viendra seulement après validation du nouveau découpage.

In [ ]:
DONNEES = '/content/drive/MyDrive/poc_ct/mmwhs'
SORTIE = '/content/drive/MyDrive/poc_ct'
!python scripts/03b_regrouper_patients.py --donnees "$DONNEES" --sortie "$SORTIE"

## 10. Étape 4 — Ligne de base TotalSegmentator (prédictions seulement, D19)
**Avant :** après un redémarrage de session, relance les sections 1 (Drive + GPU) et 2 (dépôt à jour). Le GPU doit être un **Tesla T4** : sans GPU, le script s'arrête (code 5) au lieu de tourner des heures sur le processeur.

**Ce que fait l'étape.** TotalSegmentator est un outil public déjà entraîné : on l'utilise tel quel, sans l'entraîner, sur les **5 scanners de test** (`imagesTs` : 1004, 1010, 1011, 1013, 1014). Sa tâche `heartchambers_highres` segmente 7 structures cardiaques, proches des 7 de MM-WHS. C'est la **ligne de base** : le score que nos modèles de l'étape 5 devront battre. Le script `scripts/04_totalsegmentator.py` :
- vérifie que `imagesTs` contient exactement les 5 scanners du découpage figé ;
- écrit une prédiction par scanner (`mmwhs_XXXX.nii.gz`) dans `poc_ct/predictions/totalsegmentator`, **hors de `nnunet_raw`** : un masque produit par TotalSegmentator ne doit jamais pouvoir servir à l'entraînement (licence, règle 5) ;
- écrit `tracabilite_etape4.json` (versions, tâche, options, date, durées, empreintes des 5 scanners).

**Aucun score n'est calculé avant l'étape 6** : le script ne lit aucun masque de vérité terrain (`labelsTs` reste fermé, D18) et ne fait aucune comparaison (D19).

**Licence.** La tâche `heartchambers_highres` n'est pas en libre accès : il faut une **clé de licence**, gratuite pour un usage non commercial, à demander ici : https://backend.totalsegmentator.com/license-academic/ (la clé ressemble à `aca_` suivi de chiffres). **Lis les conditions affichées par le formulaire avant de demander la clé** : vérifie que ton usage (projet personnel, non commercial, non clinique) est bien couvert, et en cas de doute écris à l'auteur avant de continuer. Sans clé, le script s'arrête (code 4) sans rien écrire.

Pour saisir la clé **sans l'écrire dans le notebook** :
- **le mieux** : icône **clé** (Secrets) dans la barre de gauche de Colab → ajouter un secret nommé `TOTALSEG_LICENCE`, valeur = ta clé, et autoriser l'accès du notebook ;
- **sinon** : la cellule te la demande dans une case masquée (`getpass`).

La clé passe au script par une variable d'environnement (jamais en argument de commande). Elle n'est jamais affichée ni enregistrée dans le dépôt, dans Drive ou dans la traçabilité ; TotalSegmentator la garde seulement dans la machine Colab, effacée en fin de session.

**Ce qui part vers les serveurs de TotalSegmentator.**
- La vérification de la licence et le téléchargement des poids du modèle cardiaque **transmettent forcément ta clé** au serveur de TotalSegmentator : c'est ainsi que fonctionne la licence. Aucune image n'est envoyée.
- Les **statistiques d'usage** (tâche, version, machine… et ton numéro de licence) sont envoyées par défaut par TotalSegmentator après chaque prédiction. Comme elles contiennent le numéro de licence, ce n'est pas anonyme : le script les **désactive par défaut**. L'option `--statistiques-usage` les réactive si tu le souhaites.
- Au premier lancement, TotalSegmentator affiche « TotalSegmentator sends anonymous usage statistics… » **même quand l'envoi est désactivé** : ce message vient du paquet et peut être ignoré. La ligne du script « Statistiques d'usage envoyées à TotalSegmentator : non » fait foi.

**Installation.** Surveille les avertissements de pip : l'installation peut changer la version de paquets déjà présents dans Colab (par exemple numpy, ou installer nnunetv2). Si pip affiche des conflits ou demande de redémarrer la session, envoie le message dans la conversation avant de lancer la cellule suivante.

**Durée et mémoire.** Au premier lancement de chaque session, TotalSegmentator télécharge ses poids (le modèle de recadrage, puis le modèle cardiaque via le serveur de licence). Pour ne pas saturer la mémoire vive de Colab (~12 Go), le script limite nnU-Net à **1 processus enfant** (option `--processus`, défaut 1 ; TotalSegmentator en lancerait 6 pour l'enregistrement). C'est un peu plus lent : compte quelques minutes par scanner (estimation, à confirmer).

**Si un message parle de mémoire pleine** (« RAM was full », « Background workers died ») ou si Colab redémarre tout seul : menu **Exécution → Redémarrer la session**, relance les sections 1, 2 et la cellule d'installation ci-dessous, puis la cellule de lancement en ajoutant `--ecraser` à la fin de la ligne `!python ...` (des prédictions partielles peuvent exister). Envoie-moi le message d'erreur.

**Résultat attendu :** « Lot de test vérifié : 5 scanners », « Statistiques d'usage envoyées à TotalSegmentator : non », puis pour chaque cas une ligne « forme identique au scanner : oui ; affine identique : oui ; étiquettes [1, 2, 3, 4, 5, 6, 7] », et enfin « Contrôle des prédictions : CONFORME ». Si le script s'arrête, le dernier message dit pourquoi (codes expliqués en tête du script) :
- **existe déjà** (code 2) : ne relance avec `--ecraser` que si tu veux vraiment remplacer les prédictions ;
- **CONTRÔLE EN ÉCHEC** (code 6) : les prédictions sont écrites mais une géométrie diffère ou une structure manque ; on l'examinera ensemble.

Copie **toute** la sortie de la cellule d'installation et de la cellule de lancement (la clé n'y apparaît pas) et envoie-la dans la conversation pour la vérification par l'agent QA.

In [ ]:
# Installation de TotalSegmentator, version épinglée (la même à chaque session, pour la traçabilité).
# Il apporte nnU-Net v2 et réutilise le PyTorch déjà installé dans Colab. À refaire à chaque nouvelle session.
!pip -q install "TotalSegmentator==2.18.0"
!pip show TotalSegmentator | head -2

In [ ]:
import os, getpass
IMAGES = '/content/drive/MyDrive/poc_ct/nnunet_raw/Dataset101_MMWHS/imagesTs'
SORTIE = '/content/drive/MyDrive/poc_ct/predictions/totalsegmentator'   # HORS de nnunet_raw (règle 5)

# Clé de licence : lue dans les secrets Colab (nom TOTALSEG_LICENCE), sinon demandée dans une case masquée.
# Elle passe au script UNIQUEMENT par une variable d'environnement (jamais en argument : un argument serait
# visible dans le code de la cellule et dans la liste des processus). Ni le notebook ni le dépôt ne la contiennent.
# Les statistiques d'usage de TotalSegmentator sont coupées par le script (option --statistiques-usage pour les réactiver).
try:
    from google.colab import userdata
    os.environ['TOTALSEG_LICENCE'] = userdata.get('TOTALSEG_LICENCE')
except Exception:
    os.environ['TOTALSEG_LICENCE'] = getpass.getpass('Clé de licence TotalSegmentator (aca_...) : ')

!python scripts/04_totalsegmentator.py --images "$IMAGES" --sortie "$SORTIE"
os.environ.pop('TOTALSEG_LICENCE', None)   # on retire la clé de la mémoire du notebook

## 11. Avant l'étape 6 — Comparer les définitions sur deux cas d'entraînement
**Pourquoi.** TotalSegmentator et MM-WHS numérotent leurs 7 structures de la même façon (1 = myocarde, …, 6 = aorte, 7 = artère pulmonaire : voir `docs/correspondance_etiquettes.md`), mais **ne les définissent pas forcément de la même façon**. Exemple : MM-WHS ne garde que l'**aorte ascendante**, de la valve jusqu'au bord supérieur des oreillettes ; TotalSegmentator segmente peut-être aussi la crosse ou l'aorte descendante. Comparer ces deux masques à l'étape 6 pénaliserait TotalSegmentator pour une différence de **définition**, pas pour une erreur. Les règles de comparaison de l'étape 6 doivent donc être fixées **avant** tout score, et **sans regarder le lot de test**.

**Ce que fait la section.** On regarde les écarts de définition sur **deux cas d'entraînement**, un de chaque famille d'examens : `mmwhs_1003` (coupes de 0,45 mm) et `mmwhs_1002` (0,625 mm). Le script `scripts/04b_comparer_definitions.py` :
- vérifie dans le découpage figé que ces deux cas sont bien en **entraînement** (un cas de test est refusé) ; il ne lit que `imagesTr` et `labelsTr`, **jamais `imagesTs` ni `labelsTs`** (D18) ;
- lance TotalSegmentator avec exactement les réglages de la section 10 (tâche `heartchambers_highres`, statistiques d'usage coupées, GPU obligatoire, 1 processus enfant) ;
- écrit les prédictions dans `poc_ct/predictions/totalsegmentator_definitions`, **hors de `nnunet_raw`**, sous un nom qui ne peut pas être confondu avec un masque d'entraînement (`mmwhs_1003_totalsegmentator.nii.gz`). Règle 5 : un masque produit par TotalSegmentator ne sert **jamais** à entraîner ;
- calcule quelques chiffres simples et dessine une figure par cas (voir plus bas). **Aucun score** (ni Dice, ni HD95) : on regarde les définitions, pas la performance.

Si les prédictions existent déjà dans Drive (cellule déjà lancée), elles sont **réutilisées** : seules les figures et les chiffres sont refaits, en une ou deux minutes, sans licence. Pour refaire les prédictions, ajouter `--ecraser` à la fin de la ligne `!python ...`.

**Avant de lancer.** Après un redémarrage de session : relance la section 1 (Drive + GPU), la section 2 (dépôt à jour), puis la **première cellule de la section 10** (installation de TotalSegmentator). La clé de licence est demandée comme en section 10 : secret Colab `TOTALSEG_LICENCE` (le mieux), sinon une case masquée. Elle n'est jamais affichée ni enregistrée. Durée : l'étape 4 a pris 576 s pour 5 scanners ; compte quelques minutes par scanner (1003 a des coupes plus fines, donc plus de coupes).

**Comment lire les figures.** Une figure par cas, une ligne par structure (1 à 7), le volume des deux sources dans le titre de la ligne. Cinq vues par ligne :
- **axiale, coronale, sagittale** : le scanner en gris, recadré autour de la structure (marge de 2 cm). **Contour bleu épais = vérité MM-WHS** ; **contour orange fin = TotalSegmentator**. Là où les deux coïncident, on voit un trait orange bordé de bleu. Les coupes passent par le centre de la structure de la vérité MM-WHS : on prend la coupe la plus proche de ce centre qui contient les deux masques. (Le centre de l'union des deux masques tomberait, pour l'aorte, entre l'aorte ascendante et l'aorte descendante.)
- **projection coronale (de face)** et **projection sagittale (de profil)** : le masque 3D est « écrasé » sur un plan, comme une ombre ; on voit d'un coup toute l'étendue de la structure. **Gris clair = les deux** ; **bleu = MM-WHS seul** ; **orange = TotalSegmentator seul**. Les traits pointillés marquent le niveau le plus haut et le plus bas de chaque source (bleu MM-WHS, orange TotalSegmentator).
- **Sens d'affichage** (lettres jaunes) : celui des radiologues, comme en section 8. **D/G** = droite/gauche du patient (la droite du patient est à **gauche** de l'image), **A/P** = avant/arrière, **H/B** = haut (tête)/bas (pieds). En sagittale, l'avant est à gauche.

**Que chercher, structure par structure.**
- **6 aorte** : de l'orange sous le bleu (aorte descendante, crosse) et « 2 morceaux » côté TotalSegmentator.
- **7 artère pulmonaire** : de l'orange au-delà de la bifurcation (début des branches droite et gauche).
- **1 myocarde** : un anneau orange autour du ventricule droit voudrait dire que TotalSegmentator inclut aussi la paroi du VD.
- **2 oreillette gauche** : départ des veines pulmonaires, auricule. **4 oreillette droite** : limite avec les veines caves (en haut et en bas). **5 ventricule droit** : paroi incluse ou non, chambre de chasse vers l'artère pulmonaire.
- **3 cavité du VG** : on s'attend à des contours proches ; c'est le témoin.

**Les chiffres** (tableau affiché par le script, et fichier `chiffres_definitions.csv`) :
- **vol MM / vol TS** : volume de la structure en mL (1 mL = 1 cm³) pour MM-WHS et TotalSegmentator ;
- **bas / haut** : niveau (en mm) de la coupe axiale la plus basse et la plus haute où la structure est présente ; un nombre plus grand est plus près de la tête ;
- **écart** = TotalSegmentator − MM-WHS, en mm. **Écart « bas » négatif** = TotalSegmentator descend plus bas ; **écart « haut » positif** = il monte plus haut ;
- **morceaux** : nombre de blocs séparés de chaque masque.

Deux cas, c'est une **indication**, pas une mesure : un écart qui apparaît sur les deux cas est probablement une différence de définition ; un écart sur un seul cas peut être une erreur ponctuelle.

**Si le script s'arrête**, le dernier message dit pourquoi : 1 = garde-fou (rien n'est écrit), 2 = prédictions incomplètes déjà présentes (relancer avec `--ecraser`), 3 = erreur pendant la prédiction, 4 = licence, 5 = pas de GPU, 6 = géométrie différente entre scanner, vérité et prédiction (la figure du cas fautif et le tableau ne sont pas faits ; la figure d'un cas traité avant peut déjà exister).

**Lire les niveaux en mm.** La valeur absolue d'un niveau dépend du repère de chaque scanner : seuls les **écarts** TotalSegmentator − MM-WHS (calculés dans le même repère pour les deux masques) sont à interpréter.

**Ce qu'il faut m'envoyer** : toute la sortie de la cellule de lancement (la clé n'y apparaît pas), les deux tableaux de chiffres, et les deux figures (captures d'écran de la dernière cellule, ou les fichiers `definitions_mmwhs_1003.png` et `definitions_mmwhs_1002.png` de Drive), ou à défaut ta description, structure par structure. Avec ces éléments, on fixera les règles de comparaison de l'étape 6 (par exemple : rapporter à part les structures définies différemment, limiter la comparaison à une zone fixée d'avance, ou ne comparer que les 4 cavités et le myocarde). Ces règles seront écrites dans le journal des décisions **avant** tout calcul de score.

In [ ]:
import os, getpass
DATASET = '/content/drive/MyDrive/poc_ct/nnunet_raw/Dataset101_MMWHS'   # seuls imagesTr et labelsTr sont lus
SORTIE = '/content/drive/MyDrive/poc_ct/predictions/totalsegmentator_definitions'   # HORS de nnunet_raw (règle 5)

# Clé de licence : comme en section 10. Lue dans les secrets Colab (nom TOTALSEG_LICENCE), sinon demandée dans
# une case masquée. Elle passe au script UNIQUEMENT par une variable d'environnement, jamais en argument.
# Inutile si les prédictions existent déjà (elles sont réutilisées) : la case peut alors rester vide (Entrée).
try:
    from google.colab import userdata
    os.environ['TOTALSEG_LICENCE'] = userdata.get('TOTALSEG_LICENCE')
except Exception:
    os.environ['TOTALSEG_LICENCE'] = getpass.getpass('Clé de licence TotalSegmentator (aca_...) : ')

# Deux cas d'ENTRAÎNEMENT, un par famille d'examens : 1003 (0,45 mm) et 1002 (0,625 mm).
!python scripts/04b_comparer_definitions.py --dataset "$DATASET" --sortie "$SORTIE" --cas 1003 1002
os.environ.pop('TOTALSEG_LICENCE', None)   # on retire la clé de la mémoire du notebook

In [ ]:
# Affiche les figures et les chiffres écrits par le script dans Drive. LECTURE SEULE : rien n'est recalculé.
import os
import pandas as pd
from IPython.display import Image, display

SORTIE = '/content/drive/MyDrive/poc_ct/predictions/totalsegmentator_definitions'
for n in ('1003', '1002'):
    figure = f'{SORTIE}/definitions_mmwhs_{n}.png'
    if os.path.exists(figure):
        print(f'mmwhs_{n} — contours : bleu = vérité MM-WHS, orange = TotalSegmentator')
        display(Image(filename=figure))
    else:
        print(f'Figure introuvable : {figure} (lancer d\'abord la cellule précédente)')

chiffres = f'{SORTIE}/chiffres_definitions.csv'
if os.path.exists(chiffres):
    # vol_ml = volume (mL) ; bas_mm / haut_mm = niveaux extrêmes (mm) ; écart = TotalSegmentator − MM-WHS
    display(pd.read_csv(chiffres))

## 12. Étape 5 — Installer nnU-Net v2 et MLflow
**Avant :** après un redémarrage de session, relance les sections 1 (Drive + GPU) et 2 (dépôt à jour).

**nnU-Net v2** est la « recette » d'entraînement de référence en segmentation médicale : il choisit lui-même ses réglages (taille des morceaux d'image, architecture du réseau, augmentations) à partir des données. On l'installe en version **2.8.1**, épinglée. C'est la version que pip avait choisie à l'étape 4 en installant TotalSegmentator 2.18.0 (qui demande seulement `nnunetv2` 2.3.1 ou plus récent).

**MLflow** est un carnet de bord des entraînements : pour chaque essai, il range les paramètres, les mesures (durée des époques, pertes, Dice) et les fichiers (journal, courbes). Ici, tout est stocké **dans ton Drive** (`poc_ct/mlflow`), sans serveur. La version légère `mlflow-skinny` suffit ; `alembic` et `sqlalchemy` lui servent à écrire sa base de données (un fichier SQLite). Leurs versions sont épinglées aussi ; `sqlalchemy` remplace la version préinstallée dans Colab, sans effet attendu sur le reste du notebook.

**À surveiller : la version de PyTorch.** PyTorch est la bibliothèque de calcul sur GPU utilisée par nnU-Net. nnU-Net 2.8.1 refuse PyTorch 2.9. Si Colab fournit cette version, pip téléchargera une autre version de PyTorch, qui peut ne pas correspondre à la version de CUDA (le pilote de calcul du GPU) installée dans Colab. On n'épingle pas PyTorch nous-mêmes : la bonne version dépend de ce CUDA, qui change d'une mise à jour de Colab à l'autre. À la place, la cellule compare la version avant et après l'installation :
- **versions identiques** : rien à faire ;
- **« PyTorch a changé de version »** : des messages de conflit avec `torchvision` ou `torchaudio` sont alors attendus. Fais **Exécution → Redémarrer la session**, relance les sections 1 et 2 (pas cette cellule), puis envoie-moi toute la sortie **avant** de lancer la section 13 : on vérifiera ensemble que le GPU est toujours utilisable.

Envoie aussi la sortie si pip affiche d'autres conflits.

In [ ]:
# Versions épinglées (les mêmes à chaque session, pour la traçabilité). À refaire à chaque nouvelle session.
import subprocess, torch
avant = torch.__version__
print('torch avant installation :', avant)
!pip -q install "nnunetv2==2.8.1" "mlflow-skinny==3.16.1" "alembic==1.20.1" "sqlalchemy==2.1.6"
!pip show nnunetv2 mlflow-skinny alembic sqlalchemy | grep -E "^(Name|Version)"
# Version de PyTorch APRÈS installation, lue par pip (l'import déjà fait garde l'ancienne en mémoire).
infos = subprocess.run(['pip', 'show', 'torch'], capture_output=True, text=True).stdout
apres = next((l.split(': ', 1)[1] for l in infos.splitlines() if l.startswith('Version:')), 'inconnue')
print('torch après installation :', apres)
if apres != avant:
    print('!!! PyTorch a changé de version : Exécution → Redémarrer la session, puis sections 1 et 2 '
          '(pas cette cellule). Envoie cette sortie dans la conversation avant de continuer.')

## 13. Étape 5 — Préparer nnU-Net : vérification, prétraitement, plis par patient
Le script `scripts/05a_preparer_nnunet.py` fait les **vérifications prévues au début de l'étape 5** (README), puis prépare l'entraînement :
1. il vérifie que `imagesTr` et `labelsTr` contiennent exactement les 15 cas d'entraînement du découpage figé, et **aucun cas de test**. Il ne regarde jamais `imagesTs` ni `labelsTs` (D18) ;
2. il construit les **5 plis de validation croisée par patient** (un pli = un groupe d'environ 3 cas mis de côté à tour de rôle pour valider) : les paires `1001/1020` et `1005/1012` restent toujours dans le même pli (D20). Les plis sont aussi **stratifiés par épaisseur de coupe**, comme le lot de test (D20). Le jeu contient deux familles d'examens, à coupes de 0,45 mm et de 0,625 mm ; stratifier, c'est répartir chaque famille séparément pour que chaque pli ressemble au jeu entier. L'épaisseur est lue dans l'en-tête des images de `imagesTr` (même définition que le script 03b). Les 3 patients d'entraînement à 0,45 mm (`1001/1020`, `1003`, `1016`) vont dans 3 plis différents, tirés au sort ; les patients à 0,625 mm complètent ensuite les plis les moins remplis. Le tirage utilise une graine fixe (42) : on retrouve les mêmes plis à chaque fois. Si les épaisseurs lues ne sont pas celles attendues, le script s'arrête (code 1) ;
3. il compare lui-même, cas par cas, **avec SimpleITK** (la bibliothèque de lecture d'images de nnU-Net), chaque image de `imagesTr` à son masque de `labelsTr` : même taille, même espacement, même origine, même orientation, avec la même tolérance que nnU-Net. Il vérifie aussi qu'aucun masque n'est vide. **C'est ce contrôle qui fait foi pour le point A3** (README) : il ne dépend pas de ce que nnU-Net affiche ;
4. il lance `nnUNetv2_plan_and_preprocess` avec `--verify_dataset_integrity`, qui refait ces vérifications à sa façon. Le script s'arrête au moindre avertissement. La commande calcule ensuite l'« empreinte » du jeu (statistiques de taille, d'espacement, d'intensité), choisit les réglages (« plans ») et **prétraite** les images (ré-échantillonnage à un espacement commun, normalisation des intensités) ;
5. seulement si tout est conforme, il écrit un **marqueur** `verification_05a.json` (vérification réussie, date, cas vérifiés), puis `splits_final.json` (les plis que nnU-Net lira) et sa fiche de traçabilité `splits_final_05a.json`, dans `poc_ct/nnunet_preprocessed/Dataset101_MMWHS/`. Au début de chaque vérification, l'ancienne fiche et l'ancien marqueur sont effacés ; en cas d'arrêt, ils ne sont pas réécrits, et le script 05b refusera de lancer.

**Pourquoi la seule configuration `3d_fullres`.** Par défaut, nnU-Net entraîne plusieurs configurations (2D coupe par coupe, 3D pleine résolution, 3D basse résolution puis affinage), 5 plis chacune, et garde la meilleure. Sur Colab gratuit, une seule est envisageable. `3d_fullres` est celle que recommandent les auteurs de nnU-Net quand on n'en entraîne qu'une. Ne prétraiter qu'elle économise aussi du temps et de la place dans Drive.

**Mémoire.** Le script limite nnU-Net à **2 processus** (programmes lancés en parallèle ; nnU-Net en lancerait 4 à 8). Colab gratuit n'a que 2 processeurs et environ 12 Go de mémoire vive : au-delà, on ne gagne rien et la mémoire sature. Si un message parle de mémoire pleine, ou si Colab redémarre tout seul : relance les sections 1, 2 et 12, puis cette cellule avec `--processus 1` ajouté à la fin de la ligne `!python ...`.

**Durée** : inconnue, probablement plusieurs dizaines de minutes (15 scanners lus depuis Drive, ré-échantillonnés, puis écrits dans Drive).

**Résultat attendu :** les 5 plis affichés avec l'épaisseur de chaque cas, 3 cas par pli, le pli de `1001/1020` et celui de `1005/1012` contenant les deux cas de la paire, au plus un patient à 0,45 mm par pli, 15 lignes « conforme » puis « Contrôle SimpleITK : CONFORME », « verify_dataset_integrity Done », puis « aucun écart signalé », « Cas prétraités en 3d_fullres : 15 » et « Résultat : PRÊT ». Si le script s'arrête, le dernier message dit pourquoi :
- **code 4, « NE PAS ENTRAÎNER »** : le contrôle SimpleITK ou nnU-Net a vu un écart de géométrie entre une image et son masque, ou un masque vide (pour nnU-Net, ce n'est souvent qu'un avertissement : le script, lui, s'arrête). C'est le point A3 de la relecture de l'étape 3 : envoie-moi le message ;
- **code 2, « existe déjà et est DIFFÉRENT »** : un autre `splits_final.json` est déjà là, peut-être tiré au hasard par nnU-Net. N'ajoute `--ecraser` qu'après en avoir parlé ;
- **prétraitement déjà fait** (cellule relancée) : ajoute `--sauter-pretraitement` pour n'écrire que les plis, sans tout recommencer. Le contrôle SimpleITK est refait (quelques minutes), et le marqueur d'une vérification réussie est exigé : s'il manque, le script s'arrête (code 1) et il faut relancer sans cette option.

Envoie **toute** la sortie dans la conversation (le journal complet de nnU-Net est aussi dans `poc_ct/nnunet_preprocessed/journal_05a_plan_and_preprocess.txt`).

In [ ]:
RAW = '/content/drive/MyDrive/poc_ct/nnunet_raw'                    # jeu écrit à l'étape 3
PRE = '/content/drive/MyDrive/poc_ct/nnunet_preprocessed'           # données prétraitées + plis
RES = '/content/drive/MyDrive/poc_ct/nnunet_results'                # modèles et points de reprise

!python scripts/05a_preparer_nnunet.py --raw "$RAW" --preprocessed "$PRE" --results "$RES"

## 14. Étape 5 — Essai à blanc : 5 époques sur le pli 0
**But : mesurer, pas encore entraîner pour de bon.** Le réglage par défaut de nnU-Net (1 000 époques × 5 plis) est impossible sur Colab gratuit. Avant de choisir un nombre d'époques, on mesure la durée d'une **époque** sur le T4. Pour nnU-Net, une époque = 250 pas d'entraînement sur des morceaux d'image (des « patchs ») tirés au hasard, puis 50 pas de validation rapide. On utilise la variante `nnUNetTrainer_5epochs` : identique au réglage par défaut, sauf qu'elle s'arrête après 5 époques. **Le résultat ne servira pas de modèle** : seulement à estimer les durées. Rien n'est décidé dans le script : on choisira ensemble la suite d'après les chiffres.

**Ce que fait `scripts/05b_essai_nnunet.py`.**
1. Il vérifie que le marqueur `verification_05a.json` de la section 13 est présent (vérification image/masque réussie), puis que `splits_final.json` existe et est **bien celui écrit par 05a** (les plis sont recalculés à partir du découpage figé, de la graine et des épaisseurs de coupe relues dans les en-têtes de `imagesTr`) : même empreinte que dans la fiche de traçabilité, et mêmes plis une fois recalculés depuis le découpage figé. Sans ce fichier, nnU-Net tirerait ses plis au hasard, cas par cas.
2. Il **copie les données prétraitées de Drive vers le disque local** de la machine Colab (`/content/nnunet_preprocessed_local`). Pendant l'entraînement, nnU-Net relit sans arrêt ces fichiers : depuis Drive, ce serait beaucoup plus lent, avec un risque d'erreurs de lecture. Les **résultats et points de reprise**, eux, sont écrits **dans Drive**, pour survivre à la fin de la session.
3. Il lance `nnUNetv2_train 101 3d_fullres 0 -tr nnUNetTrainer_5epochs`. À la fin, nnU-Net fait sa **validation finale** : il segmente en entier les 3 scanners de validation du pli 0 et calcule leur Dice. Ces 3 cas font partie des 15 d'entraînement : **le lot de test n'est jamais utilisé**.
4. Il lit le **journal d'entraînement** de nnU-Net (`training_log_*.txt`). Il vérifie que nnU-Net a écrit « Using splits from existing split file » avec le bon fichier, et que les 3 scanners prédits en validation finale sont bien ceux du pli 0. Il affiche pour chaque époque la durée, les pertes et le **Dice pseudo** (Dice approximatif, calculé sur quelques patchs : seulement une tendance). Il affiche aussi la durée moyenne **sans la 1re époque** (plus lente : compilation, mise en route) et une **extrapolation indicative** pour 100, 250 et 1 000 époques. La durée « Epoch time » de nnU-Net n'inclut pas l'écriture de `checkpoint_best.pth` dans Drive : le script affiche ce temps « hors époque » à part, à ajouter à l'extrapolation.
5. Il trace l'essai dans **MLflow** : paramètres (pli, trainer, configuration, versions, GPU, version du code), durées, pertes, Dice pseudo, et le journal en pièce jointe.

**Risques pratiques, et réglages proposés** (options à ajouter à la fin de la ligne `!python ...`) :
- **Lectures depuis Drive** : évitées par la copie locale (par défaut). `--lire-depuis-drive` la désactive (déconseillé).
- **torch.compile** : par défaut, nnU-Net fait traduire le réseau par PyTorch en code optimisé pour le GPU. Ensuite, c'est plus rapide, mais la compilation ajoute plusieurs minutes à la 1re époque, et elle peut échouer sur certaines combinaisons de GPU et de versions. Si l'essai plante avec un message qui parle de `torch._dynamo`, `inductor` ou `triton`, relance avec `--ecraser --sans-compile`. On garde le réglage par défaut d'abord, pour que la mesure soit représentative.
- **Mémoire vive** : nnU-Net prépare les patchs (augmentation de données) dans des processus parallèles. Il en lance au plus autant que de processeurs, soit 2 sur Colab gratuit : c'est probablement ce qui limitera la vitesse, plus que le GPU. L'export de la validation finale est limité à 2 processus (`--processus 1` si la mémoire sature).
- **Validation finale** : sa durée est mesurée à part (3 volumes entiers, à pleine résolution). Elle s'ajoutera à chaque pli de l'entraînement définitif.
- **Session coupée** : Colab gratuit coupe au bout de quelques heures, ou plus tôt si l'onglet reste inactif. nnU-Net écrit `checkpoint_best.pth` (meilleur modèle) à chaque progrès, et `checkpoint_latest.pth` seulement **toutes les 50 époques** : une coupure peut donc faire perdre jusqu'à 49 époques. Pour **reprendre** : relance les sections 1, 2 et 12, puis cette cellule avec `--continuer` (option `--c` de nnU-Net, qui repart du dernier point de reprise). Si l'entraînement s'est terminé mais que le traçage MLflow a manqué : `--tracer-seulement`.

**Durée** : c'est ce que l'essai mesure. Prévois au moins une heure de session.

**Résultat attendu :** « splits_final.json vérifié », la copie locale, la sortie de nnU-Net (5 époques), puis le tableau des époques, la durée moyenne, l'extrapolation, « Essai tracé dans MLflow » et « Résultat : ESSAI TERMINÉ ». Codes d'arrêt : 1 = garde-fou (rien n'est lancé) ; 2 = un essai existe déjà (`--continuer` ou `--ecraser`) ; 3 = nnU-Net s'est arrêté en erreur ; 5 = pas de GPU ; **6 = le journal ne confirme pas que nnU-Net a utilisé nos plis : ne pas utiliser les résultats**.

Envoie **toute** la sortie de la cellule dans la conversation, ainsi que le tableau et la courbe affichés par la cellule suivante.

In [ ]:
RAW = '/content/drive/MyDrive/poc_ct/nnunet_raw'                    # jeu écrit à l'étape 3
PRE = '/content/drive/MyDrive/poc_ct/nnunet_preprocessed'           # données prétraitées + plis
RES = '/content/drive/MyDrive/poc_ct/nnunet_results'                # modèles et points de reprise
MLFLOW = '/content/drive/MyDrive/poc_ct/mlflow'                     # carnet de bord MLflow
!python scripts/05b_essai_nnunet.py --raw "$RAW" --preprocessed "$PRE" --results "$RES" --mlflow "$MLFLOW"

In [ ]:
# Affiche les essais enregistrés dans MLflow et la courbe d'apprentissage de nnU-Net. LECTURE SEULE.
# La base de Drive est copiée avant lecture : elle n'est jamais ouverte directement (verrous de fichiers sur Drive).
import os, shutil, tempfile
import mlflow
from IPython.display import Image, display

MLFLOW = '/content/drive/MyDrive/poc_ct/mlflow'
RES = '/content/drive/MyDrive/poc_ct/nnunet_results'
if not os.path.exists(f'{MLFLOW}/mlflow.db'):
    print(f'Aucune base MLflow dans {MLFLOW} : lancer d\'abord la cellule précédente jusqu\'à « Essai tracé dans MLflow ».')
else:
    copie = os.path.join(tempfile.mkdtemp(), 'mlflow.db')
    shutil.copy2(f'{MLFLOW}/mlflow.db', copie)
    mlflow.set_tracking_uri('sqlite:///' + copie)
    essais = mlflow.search_runs(experiment_names=['etape5_nnunet'])
    colonnes = ['start_time', 'tags.statut', 'params.trainer', 'params.pli', 'params.gpu', 'params.torch_compile',
                'metrics.nb_epoques_terminees', 'metrics.duree_1re_epoque_s', 'metrics.duree_epoque_moyenne_sans_1re_s',
                'metrics.hors_epoque_s', 'metrics.validation_finale_s', 'metrics.dice_validation_finale',
                'metrics.extrapolation_1000_epoques_h_par_pli']
    display(essais[[c for c in colonnes if c in essais.columns]])

# progress.png : courbes de nnU-Net (pertes, Dice pseudo, durée des époques, taux d'apprentissage).
courbe = f'{RES}/Dataset101_MMWHS/nnUNetTrainer_5epochs__nnUNetPlans__3d_fullres/fold_0/progress.png'
if os.path.exists(courbe):
    display(Image(filename=courbe))
else:
    print('Courbe introuvable :', courbe)

## 15. Étape 5 — Entraînement d'un pli : 250 époques, reprise automatique (D25)
**But : entraîner pour de bon**, un pli à la fois. D'après l'essai de la section 14 et la décision D25 : configuration `3d_fullres`, **250 époques par pli** (au lieu de 1 000 par défaut), sur GPU **L4**. Une époque y dure environ 115 s : un pli demande donc **environ 8 h et 15 à 16 unités de calcul**, plus la validation finale. On commence par le **pli 0** ; on ne passe au suivant qu'une fois le pli terminé et ses résultats regardés ensemble.

**Choisir le pli.** Dans la cellule ci-dessous, change seulement la ligne `PLI = 0`. Les 3 cas de validation de chaque pli ont été fixés à la section 13 (D24) : le script vérifie qu'ils n'ont pas changé.

**La variante d'entraînement `nnUNetTrainer_250epochs_reprise10`** (fichier `scripts/trainers_nnunet/`). Un *trainer* est la classe de nnU-Net qui conduit l'entraînement. Celle-ci reprend le trainer par défaut et ne change que trois choses :
- 250 époques au lieu de 1 000 ;
- un **point de reprise** toutes les **10** époques au lieu de 50. Un point de reprise (`checkpoint_latest.pth`) est un fichier qui contient tout l'état de l'entraînement : les poids du réseau, l'état de l'optimiseur, le numéro de l'époque et l'historique des courbes. Si la session est coupée, on perd au plus 9 époques (environ 17 min) ;
- ce fichier est écrit d'abord sous un nom provisoire, puis renommé une fois complet : une coupure pendant l'écriture ne peut pas abîmer le point de reprise précédent.

Le nom du trainer fait partie du dossier de résultats : `nnunet_results/Dataset101_MMWHS/nnUNetTrainer_250epochs_reprise10__nnUNetPlans__3d_fullres/fold_0/`. Il ne faut donc pas le changer en cours de route.

**Reprise après une coupure : relancer la même cellule.** Colab peut couper la session (durée maximale, inactivité, unités épuisées). Les points de reprise sont écrits **dans Drive** : ils survivent. Pour reprendre, ouvre une nouvelle session, relance les sections **1, 2 et 12**, puis **cette même cellule, sans rien changer**. Le script regarde lui-même où en est le pli :
- aucun point de reprise : il démarre un nouvel entraînement ;
- un point de reprise : il **reprend** (option `--c` de nnU-Net) et affiche l'époque de reprise, les époques restantes, le temps et les unités estimés ;
- `checkpoint_final.pth` et la validation finale présents : le pli est **terminé**, rien n'est relancé, le suivi MLflow est seulement mis à jour.

Tu n'as donc jamais à choisir entre « démarrer » et « reprendre ». Seul cas particulier : si la coupure tombe pendant la validation finale (dernière demi-heure environ), le script s'arrête avec le code 2 et te demande d'ajouter `--terminer-validation` ; aucune époque n'est alors réentraînée.

**Conseils Colab.**
- **Choisir le L4** : Exécution → Modifier le type d'exécution → GPU **L4**. Sur un autre GPU, le script avertit sans bloquer : l'entraînement marche, mais les durées et le coût de D25 ne valent plus (un T4 est environ 3,7 fois plus lent).
- **Garder l'onglet ouvert** et l'ordinateur allumé, sans mise en veille : Colab coupe les sessions jugées inactives. Fermer l'onglet ou laisser l'ordinateur s'endormir peut arrêter l'entraînement (il reprendra au dernier point de reprise).
- **Vérifier les unités restantes** avant de lancer (icône de ressources en haut à droite) : il en faut environ 16 pour un pli complet. Sinon, la session s'arrêtera en route ; ce n'est pas grave, on reprendra plus tard.
- **Arrêter volontairement** : bouton « stop » de la cellule. Le script arrête nnU-Net, met à jour MLflow et s'arrête (code 3). On reprend en relançant la cellule.
- **Une seule session à la fois par pli.** Pendant l'entraînement, le script tient à jour un fichier verrou (`entrainement_en_cours.verrou`) dans le dossier du pli. Si une autre session essaie d'entraîner le même pli, elle s'arrête (code 1). Après une coupure brutale, le verrou reste quelques minutes : si le script refuse de reprendre, attends 15 minutes, ou, si tu es **sûr** qu'aucune autre session ne tourne (un seul onglet Colab ouvert), ajoute `--ignorer-verrou` à la fin de la ligne `!python ...`.

**Suivi pendant l'entraînement.** La sortie de nnU-Net s'affiche au fil des époques. Toutes les 30 minutes, le script relit le journal et met à jour **MLflow** dans Drive (un seul *run* par pli, complété à chaque session : on y retrouve toutes les époques, quelles que soient les coupures). Seules les époques déjà couvertes par un point de reprise y sont notées (elles ne seront jamais refaites) : le suivi MLflow a donc jusqu'à 10 époques de retard sur la sortie de nnU-Net, c'est normal.

**Regarder la courbe pendant l'entraînement.** Colab exécute les cellules d'une session **l'une après l'autre** : tant que l'entraînement tourne, la cellule de suivi attendrait son tour. Pendant l'entraînement, ouvre plutôt `progress.png` directement dans Google Drive (dossier `poc_ct/nnunet_results/Dataset101_MMWHS/nnUNetTrainer_250epochs_reprise10__nnUNetPlans__3d_fullres/fold_0/`, mis à jour à chaque époque). La cellule de suivi se lance **après coup** (fin de l'entraînement, ou nouvelle session).

**Ce qui n'est jamais fait** : aucune lecture de `imagesTs` ni `labelsTs`, aucune prédiction sur le lot de test (étape 6). Le « Dice de validation finale » affiché porte sur les 3 cas de validation du pli, qui font partie des 15 cas d'entraînement.

**À savoir pour l'étape 6** : pour prédire avec ce modèle, nnU-Net devra retrouver la variante d'entraînement par son nom. Il faudra donc fixer la même variable `nnUNet_extTrainer` (vers `scripts/trainers_nnunet`). Le script de l'étape 6 s'en chargera.

**Résultat attendu :** « splits_final.json vérifié », la copie locale, « NOUVEL ENTRAÎNEMENT » ou « REPRISE depuis ... : époque N sur 250 », la sortie de nnU-Net, des lignes « MLflow mis à jour » toutes les 30 min, puis le tableau des 10 dernières époques, la durée moyenne d'une époque, le temps et les unités restants, et « Résultat : pli 0 TERMINÉ » (ou « en cours » si la session s'est arrêtée). Codes d'arrêt : 1 = garde-fou, dont le verrou d'une autre session (rien n'est lancé) ; 2 = validation finale incomplète (`--terminer-validation`) ; 3 = session interrompue ou erreur de nnU-Net (relancer la même cellule) ; 5 = pas de GPU ; **6 = le journal ne confirme pas que nnU-Net a utilisé nos plis : ne pas utiliser les résultats**. Si l'erreur parle de `torch._dynamo`, `inductor` ou `triton`, ajoute `--sans-compile` à la fin de la ligne `!python ...`.

**Copie de la sortie dans Drive** : tout ce que le script affiche, y compris la sortie de nnU-Net, est aussi recopié dans `fold_X/sortie_05c_<SESSION>.txt`, un fichier par session (SESSION est la date et l'heure UTC du lancement). Après une coupure, la sortie de la session reste donc lisible. Les heures sont affichées en UTC (l'heure de la machine Colab) et en heure de Paris.

Envoie la **fin** de la sortie (à partir de « nnU-Net terminé » ou « interrompu ») dans la conversation, avec ce qu'affiche la cellule de suivi.

In [ ]:
PLI = 0                                                             # pli à entraîner : 0, puis 1, 2, 3, 4 (D25)
RAW = '/content/drive/MyDrive/poc_ct/nnunet_raw'                    # jeu écrit à l'étape 3
PRE = '/content/drive/MyDrive/poc_ct/nnunet_preprocessed'           # données prétraitées + plis
RES = '/content/drive/MyDrive/poc_ct/nnunet_results'                # modèles et points de reprise
MLFLOW = '/content/drive/MyDrive/poc_ct/mlflow'                     # carnet de bord MLflow
# Même cellule pour démarrer ET pour reprendre après une coupure : le script détecte où en est le pli.
!python scripts/05c_entrainer_nnunet.py --pli $PLI --raw "$RAW" --preprocessed "$PRE" --results "$RES" --mlflow "$MLFLOW"

In [ ]:
# Suivi de l'entraînement : runs MLflow, points de reprise et courbe du pli choisi. LECTURE SEULE.
# Se lance APRÈS l'entraînement (Colab exécute les cellules l'une après l'autre) ; pendant, ouvrir
# progress.png dans Drive. La base de Drive est copiée avant lecture : elle n'est jamais ouverte directement.
# Heures : la machine Colab est en UTC ; l'heure de Paris est affichée à côté.
import datetime, glob, os, shutil, tempfile
import mlflow
from IPython.display import Image, display
try:
    from zoneinfo import ZoneInfo
    PARIS, NOM_PARIS = ZoneInfo('Europe/Paris'), 'Paris'
except Exception:                                                   # base des fuseaux absente : heure d'hiver supposée
    PARIS, NOM_PARIS = datetime.timezone(datetime.timedelta(hours=1)), 'Paris si UTC+1'

PLI = 0                                                             # même pli que la cellule précédente
MLFLOW = '/content/drive/MyDrive/poc_ct/mlflow'
RES = '/content/drive/MyDrive/poc_ct/nnunet_results'
TRAINER = 'nnUNetTrainer_250epochs_reprise10'
dossier = f'{RES}/Dataset101_MMWHS/{TRAINER}__nnUNetPlans__3d_fullres/fold_{PLI}'

if not os.path.exists(f'{MLFLOW}/mlflow.db'):
    print(f'Aucune base MLflow dans {MLFLOW} : elle apparaît au premier « MLflow mis à jour » (30 min après le lancement).')
else:
    copie = os.path.join(tempfile.mkdtemp(), 'mlflow.db')
    shutil.copy2(f'{MLFLOW}/mlflow.db', copie)
    mlflow.set_tracking_uri('sqlite:///' + copie)
    runs = mlflow.search_runs(experiment_names=['etape5_nnunet'])
    if 'params.trainer' in runs.columns:
        runs = runs[runs['params.trainer'] == TRAINER]
    if 'start_time' in runs.columns:                                # start_time est en UTC ; ajout de l'heure de Paris
        try:
            runs = runs.assign(debut_paris=runs['start_time'].dt.tz_convert(PARIS).dt.strftime('%d/%m %H:%M'))
        except Exception:
            pass
    colonnes = ['params.pli', 'tags.statut', 'start_time', 'debut_paris', 'metrics.epoques_terminees',
                'metrics.duree_epoque_moyenne_s', 'metrics.epoques_consolidees', 'metrics.duree_epoque_reelle_s',
                'metrics.dice_validation_finale', 'params.val_pli']
    display(runs[[c for c in colonnes if c in runs.columns]])

# Points de reprise présents dans Drive (date de la dernière écriture, en UTC et heure de Paris).
for nom in ('checkpoint_latest.pth', 'checkpoint_best.pth', 'checkpoint_final.pth'):
    chemin = f'{dossier}/{nom}'
    if os.path.exists(chemin):
        t = datetime.datetime.fromtimestamp(os.path.getmtime(chemin), datetime.timezone.utc)
        print(f'{nom} : écrit le {t:%d/%m %H:%M} UTC ({t.astimezone(PARIS):%d/%m %H:%M} {NOM_PARIS}), '
              f'{os.path.getsize(chemin) / 1e6:.0f} Mo')
    else:
        print(f'{nom} : absent')

# Copies de l'affichage de 05c, une par session (nom = date et heure UTC du lancement).
sorties = sorted(glob.glob(f'{dossier}/sortie_05c_*.txt'))
print('Sorties de 05c :', ', '.join(os.path.basename(s) for s in sorties) if sorties else 'aucune')

# progress.png : courbes de nnU-Net (pertes, Dice pseudo, durée des époques, taux d'apprentissage).
courbe = f'{dossier}/progress.png'
if os.path.exists(courbe):
    display(Image(filename=courbe))
else:
    print('Courbe introuvable :', courbe)

## 16. Étape 5 — Analyser l'échec du pli 0 sur les vaisseaux de mmwhs_1019 (D26)
**Avant :** après un redémarrage de session, relance les sections 1 (Drive) et 2 (dépôt à jour). **Pas besoin de GPU** ni de la section 12 : `nibabel`, `numpy`, `scipy` et `matplotlib` sont déjà installés dans Colab (si `nibabel` manque, relance la section 3).

**Pourquoi cette analyse.** Le pli 0 de nnU-Net a un Dice moyen de validation de 0,850 sur ses 3 cas (1001, 1019, 1020). Ce chiffre cache un échec complet sur `mmwhs_1019` : **aorte (étiquette 6) Dice 0,00** (378 433 voxels dans la vérité, 14 784 prédits) et **artère pulmonaire (7) 0,46**, alors que les cavités et le myocarde y sont entre 0,89 et 0,96, et que l'aorte est à 0,97-0,98 sur 1001 et 1020. Avant de dépenser environ 16 unités de calcul par pli (D25), on veut savoir si le problème vient **du cas** ou **du modèle**. Trois hypothèses :
- **(a) cadrage du scanner** : le champ de vue (la zone du corps couverte par le scanner) est différent, ou l'aorte est coupée par le bord de l'image ;
- **(b) qualité de l'image** : le produit de contraste (le liquide injecté qui rend le sang blanc) est faible dans l'aorte, ou l'image est très bruitée (granuleuse) ;
- **(c) annotation** : la vérité terrain dit « aorte » sur autre chose que l'aorte ascendante de la définition MM-WHS (aorte descendante incluse, autre vaisseau, étiquettes 6 et 7 inversées...).

**Ce que fait `scripts/05d_analyser_1019.py`**, sur 1019 et, pour comparer, sur 1001 (cas réussi) :
1. **Géométrie** : taille de l'image, taille des voxels (espacement, en mm), épaisseur de coupe, orientation, étendue du champ en mm sur chaque axe. Il lit aussi l'**en-tête seul** (sans les images) des 15 cas d'entraînement, pour voir si le champ de 1019 est inhabituel.
2. **Aorte et artère pulmonaire**, dans la vérité **et** dans la prédiction : nombre de voxels, volume (mL), première et dernière coupe qui les contiennent (en indice et en mm), boîte englobante (le plus petit pavé qui contient la structure), distance aux bords de l'image, **centre de gravité** (position moyenne de la structure, en mm) et décalage vérité/prédiction, nombre de **morceaux** (blocs séparés).
3. **Matrice de confusion** : pour chaque voxel de la vérité « aorte », quelle étiquette la prédiction lui a donnée (fond ? artère pulmonaire ? cavité ?), et inversement, sur quoi tombe l'« aorte » prédite. C'est l'indicateur clé : « aorte vue comme artère pulmonaire » (confusion d'étiquettes) et « aorte vue comme fond » (non détectée) ne racontent pas la même histoire.
4. **Intensités** en **HU** (unités Hounsfield, l'échelle des scanners : eau = 0, air = −1000, sang injecté autour de 250 à 500) dans l'aorte et l'artère pulmonaire de la vérité, comparées à la cavité du ventricule gauche (VG, étiquette 3, vérifiée dans `dataset.json`) et au ventricule droit (VD). Le sang de l'aorte vient du VG : ils devraient avoir à peu près la même intensité ; de même pour l'artère pulmonaire et le VD. Il mesure aussi le **bruit** (écart-type des HU au centre du VG, une zone qui devrait être uniforme) et le **CNR** (rapport contraste sur bruit : écart sang/myocarde divisé par le bruit ; plus il est grand, plus les bords sont nets).
5. Une **figure** par cas (voir ci-dessous).
6. Un **récapitulatif** rangé par hypothèse : pour chaque indicateur, la valeur sur 1019 et sur 1001, et ce qu'une valeur inhabituelle suggérerait. **Le script ne conclut pas** : on décide ensemble, sur les chiffres et les figures.

**Lecture seule, et jamais le lot de test.** Le script lit seulement `imagesTr`, `labelsTr` et `dataset.json` de `Dataset101_MMWHS`, et les prédictions de `nnunet_results/.../nnUNetTrainer_250epochs_reprise10__nnUNetPlans__3d_fullres/fold_0/validation/`. Il **refuse** tout cas qui n'est pas en entraînement dans le découpage figé `splits/mmwhs_split.json` (lu, jamais modifié) : les cas de test 1004, 1010, 1011, 1013 et 1014 sont rejetés (code 1). Il n'écrit **que** dans `poc_ct/analyses/05d_1019/` (figures PNG et `synthese_05d.json`), et s'arrête si ce dossier est placé dans un dossier nnU-Net ou dans le dépôt.

**Lire la figure.** Deux rangées : la 1re passe par le centre de l'aorte de la vérité, la 2e par celui de l'artère pulmonaire. Colonnes : coupes **axiale** (vue par les pieds), **coronale** (de face), **sagittale** (de profil), puis une **projection de face** (les masques « écrasés » sur un plan, comme une ombre ; en gris, tout le cœur de la vérité). L'image n'est pas recadrée : **ses bords sont les bords du champ du scanner**, on voit donc si un vaisseau est coupé. Contours : **rouge** = vérité aorte, **jaune** = aorte prédite, **bleu** = vérité artère pulmonaire, **magenta** = artère pulmonaire prédite ; trait épais = vérité, trait fin = prédiction. Pointillé blanc = niveau de la coupe axiale. Fenêtre d'affichage −200 / 600 HU (adaptée au sang injecté). Lettres jaunes : D/G = droite/gauche du patient (sa droite est à gauche de l'image, comme en radiologie), A/P = avant/arrière, H/B = haut/bas.

**Point d'attention.** 1001 (et 1020, même patient) est le seul cas réussi qui a une prédiction, et il appartient à l'autre famille d'examens (coupes de 0,45 mm, probablement un autre scanner) que 1019. Un écart de bruit ou de contraste entre 1019 et 1001 peut donc venir de la famille d'examens, pas seulement du cas. Pour ajouter 1020 à la comparaison : `--cas 1019 1001 1020`.

**Durée** : quelques minutes (2 scanners lus depuis Drive).

**Résultat attendu :** « Cas vérifiés : ... en ENTRAÎNEMENT », « Étiquettes de dataset.json conformes », puis pour chaque cas les tableaux 1 à 4 et « Figure : ... », le champ de vue des 15 cas d'entraînement, le récapitulatif « 6. RÉCAPITULATIF » et « Synthèse : ... ». Une ligne **« ANOMALIE »** signale un en-tête différent entre image, vérité et prédiction (l'analyse continue, voxel à voxel). Codes d'arrêt : 1 = garde-fou (cas refusé, fichier introuvable, dossier de sortie interdit, étiquettes inattendues ; rien n'est écrit) ; 6 = image, vérité et prédiction n'ont pas la même taille.

Envoie **toute** la sortie de la cellule dans la conversation, ainsi que les deux figures et le tableau affichés par la cellule suivante.

In [ ]:
RAW = '/content/drive/MyDrive/poc_ct/nnunet_raw'                    # seuls imagesTr, labelsTr et dataset.json sont lus
RES = '/content/drive/MyDrive/poc_ct/nnunet_results'                # seul fold_0/validation est lu
SORTIE = '/content/drive/MyDrive/poc_ct/analyses/05d_1019'          # HORS des dossiers nnU-Net : figures + synthèse
# LECTURE SEULE. Premier cas = cas étudié ; les suivants = cas de comparaison (1001 : aorte réussie).
# Tout cas hors du lot d'entraînement du découpage figé est refusé (test : 1004, 1010, 1011, 1013, 1014).
!python scripts/05d_analyser_1019.py --raw "$RAW" --results "$RES" --pli 0 --cas 1019 1001 --sortie "$SORTIE"

In [ ]:
# Affiche les figures et le récapitulatif écrits par le script dans Drive. LECTURE SEULE : rien n'est recalculé.
import json, os
import pandas as pd
from IPython.display import Image, display

SORTIE = '/content/drive/MyDrive/poc_ct/analyses/05d_1019'
for n in ('1019', '1001'):
    figure = f'{SORTIE}/analyse_mmwhs_{n}.png'
    if os.path.exists(figure):
        print(f'mmwhs_{n} — rouge = vérité aorte, jaune = aorte prédite, bleu = vérité AP, magenta = AP prédite')
        display(Image(filename=figure))
    else:
        print(f'Figure introuvable : {figure} (lancer d\'abord la cellule précédente)')

# Récapitulatif par hypothèse : (a) cadrage, (b) qualité de l'image, (c) annotation. Une colonne par cas.
synthese = f'{SORTIE}/synthese_05d.json'
if os.path.exists(synthese):
    with open(synthese, encoding='utf-8') as f:
        s = json.load(f)
    lignes = [{'hypothèse': h, 'indicateur': l['libelle'],
               **{cas[-4:]: ind[h][cle] for cas, ind in s['indicateurs'].items()},
               'si la valeur est inhabituelle': l['si']}
              for h, d in s['lecture'].items() for cle, l in d.items()]
    pd.set_option('display.max_colwidth', None)
    display(pd.DataFrame(lignes))

## 17. Étape 5 — Diagnostic de l'incident de reprise du pli 0 (lecture seule)

Après une coupure de la session vers 15 h 52, l'entraînement a repris à l'époque 200, alors qu'on le pensait vers l'époque 230. Cette cellule ne modifie rien. Elle lit les fichiers de `fold_0`, les points de reprise, les journaux nnU-Net et la base MLflow pour répondre à deux questions : les points de reprise 210 à 230 ont-ils été écrits, et pourquoi la reprise est-elle repartie de 200 ?

Deux hypothèses principales :
- **H1** : l'entraînement n'avait pas encore atteint l'époque 210. Colab et nnU-Net affichent l'heure **UTC**, soit une heure de moins qu'à Paris : une heure, c'est environ 31 époques, presque exactement l'écart entre 200 et 230.
- **H2** : les points de reprise 210 à 230 ont été écrits sur le disque de Colab, mais Google Drive n'avait pas fini de les envoyer vers ses serveurs quand la machine s'est arrêtée.

Ce qu'il faut regarder :
- l'heure de fin de l'époque 199 (section 3) ;
- la dernière époque terminée dans le journal de la session coupée (section 4) ;
- les époques notées deux fois dans MLflow (section 5).

Avant de lancer, vérifiez la date de la variable `COUPURE_PARIS`. Ce diagnostic n'a pas besoin du GPU.

In [ ]:
# 17. Diagnostic de l'incident de reprise du pli 0 (reprise à l'époque 200 au lieu d'environ 230). LECTURE SEULE.
# Rien n'est écrit : ni dans Drive, ni dans le dépôt, ni sur le disque local. La base MLflow est ouverte
# en lecture seule (mode=ro, immutable=1), sans copie. Les points de reprise sont lus sur le processeur.
# Heures : la machine Colab est en UTC ; les journaux nnU-Net sont donc en UTC. Paris = UTC+1 en novembre.
import datetime, glob, os, re, sqlite3, statistics, urllib.parse
from zoneinfo import ZoneInfo
import torch

PLI = 0
RES = '/content/drive/MyDrive/poc_ct/nnunet_results'
MLFLOW = '/content/drive/MyDrive/poc_ct/mlflow'
TRAINER = 'nnUNetTrainer_250epochs_reprise10'
COUPURE_PARIS = '2026-11-10 15:52'        # heure de la coupure, HEURE DE PARIS (corriger la date si besoin)
FENETRE_MIN = 45                          # minutes affichées avant et après la coupure
dossier = f'{RES}/Dataset101_MMWHS/{TRAINER}__nnUNetPlans__3d_fullres/fold_{PLI}'
UTC = datetime.timezone.utc
try:
    PARIS = ZoneInfo('Europe/Paris')
except Exception:                                        # base des fuseaux absente : heure d'hiver fixe (UTC+1)
    PARIS = datetime.timezone(datetime.timedelta(hours=1), 'Paris (UTC+1)')


def heure(t):
    """Horodatage (secondes ou datetime) → « jj/mm HH:MM:SS UTC = HH:MM:SS Paris »."""
    d = t if isinstance(t, datetime.datetime) else datetime.datetime.fromtimestamp(t, UTC)
    return f'{d.astimezone(UTC):%d/%m %H:%M:%S} UTC = {d.astimezone(PARIS):%H:%M:%S} Paris'


coupure = datetime.datetime.fromisoformat(COUPURE_PARIS).replace(tzinfo=PARIS)
print(f'Coupure annoncée : {heure(coupure)}\nDossier du pli : {dossier}\n')

# --- 1. Fichiers du pli (date de dernière modification vue par le montage Drive, taille) -------------
print('=== 1. Fichiers du dossier du pli (hors sous-dossier validation/) ===')
for nom in sorted(os.listdir(dossier)):
    c = os.path.join(dossier, nom)
    if os.path.isfile(c):
        st = os.stat(c)
        print(f'{nom:<58} {st.st_size / 1e6:>9.1f} Mo   modifié {heure(st.st_mtime)}')
restes = sorted(glob.glob(f'{dossier}/*.ecriture_en_cours') + glob.glob(f'{dossier}/*.tmp'))
print('Fichiers temporaires restants (.ecriture_en_cours, .tmp) :', restes or 'aucun')
print('Verrou entrainement_en_cours.verrou présent :', os.path.exists(f'{dossier}/entrainement_en_cours.verrou'))
print('NB : à la fin d\'un entraînement TERMINÉ, nnU-Net supprime checkpoint_latest.pth : son absence est normale.')

# --- 2. Points de reprise : époque notée et longueur de l'historique (logger) -------------------------
print('\n=== 2. Points de reprise ===')
historiques = {}
for nom in ('checkpoint_latest.pth', 'checkpoint_best.pth', 'checkpoint_final.pth'):
    c = f'{dossier}/{nom}'
    if not os.path.isfile(c):
        print(f'{nom} : absent')
        continue
    ck = torch.load(c, map_location='cpu', weights_only=False)
    log = ck.get('logging', {})
    print(f"{nom} : current_epoch = {ck['current_epoch']} (époques 0 à "
          f"{ck['current_epoch'] - 1} faites ; prochaine : {ck['current_epoch']}) ; _best_ema = {ck.get('_best_ema')} ; trainer = {ck.get('trainer_name')}")
    print('   longueur des listes du logger :', {k: len(v) for k, v in log.items()})
    historiques[nom] = log
    del ck                                               # libère la mémoire (fichiers de plusieurs centaines de Mo)

# --- 3. Chronologie réelle des époques, d'après les horodatages gardés dans le point de reprise ------
# Le logger de nnU-Net garde l'heure de début et de fin de CHAQUE époque. Une reprise depuis l'époque N
# conserve les époques 0..N-1 de la session coupée : la fin de l'époque 199 est donc l'heure d'écriture
# du point de reprise « 200 », même si checkpoint_latest.pth a été supprimé depuis.
print('\n=== 3. Chronologie des époques (horodatages du logger) ===')
nom_ref = 'checkpoint_final.pth' if 'checkpoint_final.pth' in historiques else 'checkpoint_best.pth'
log = historiques.get(nom_ref)
if log and log.get('epoch_start_timestamps') and log.get('epoch_end_timestamps'):
    debuts, fins = log['epoch_start_timestamps'], log['epoch_end_timestamps']
    n = min(len(debuts), len(fins))
    print(f'Référence : {nom_ref} ({n} époques). Époque 0 : début {heure(debuts[0])} ; '
          f'époque {n - 1} : fin {heure(fins[n - 1])}')
    duree = statistics.median(b - a for a, b in zip(debuts[1:n], debuts[2:n])) if n > 3 else None
    if duree:
        print(f'Durée réelle médiane d\'une époque : {duree:.1f} s')
    for k in range(1, n):                                # un trou anormal = une nouvelle session (reprise)
        trou = debuts[k] - fins[k - 1]
        if trou > 300:
            print(f'  REPRISE à l\'époque {k} : fin de l\'époque {k - 1} {heure(fins[k - 1])} -> '
                  f'début de l\'époque {k} {heure(debuts[k])} (trou de {trou / 60:.0f} min)')
    for k in range(9, n, 10):                            # instants d'écriture de checkpoint_latest
        if 189 <= k <= 239:
            print(f'  fin de l\'époque {k} (checkpoint_latest current_epoch={k + 1} écrit juste après) : {heure(fins[k])}')
    faites = [k for k in range(n) if fins[k] <= coupure.timestamp()]
    if faites:
        k = max(faites)
        print(f'Dans la branche GARDÉE, dernière époque finie avant la coupure : {k} ({heure(fins[k])}).')
        if duree:
            print(f'  Borne haute : même si la session coupée avait tourné sans arrêt jusqu\'à {COUPURE_PARIS} (Paris), '
                  f'elle n\'aurait pas dépassé l\'époque {k + (coupure.timestamp() - fins[k]) / duree:.0f} '
                  '(la dernière époque réellement atteinte est dans son journal, section 4).')
else:
    print('Aucun historique d\'horodatages lisible.')

# checkpoint_best vient-il de la branche gardée ou de la branche abandonnée (époques refaites) ?
if {'checkpoint_best.pth', 'checkpoint_final.pth'} <= set(historiques):
    b = historiques['checkpoint_best.pth'].get('epoch_start_timestamps', [])
    f = historiques['checkpoint_final.pth'].get('epoch_start_timestamps', [])
    ecarts = [k for k in range(min(len(b), len(f))) if b[k] != f[k]]
    print('checkpoint_best.pth :', 'même branche que checkpoint_final.pth' if not ecarts else
          f'BRANCHE ABANDONNÉE (diffère de final dès l\'époque {ecarts[0]}) : sans effet sur la validation, '
          'faite avec checkpoint_final')

# --- 4. Journaux nnU-Net : un par session --------------------------------------------------------------
print('\n=== 4. Journaux nnU-Net (heures des lignes = UTC) ===')
RE_LIGNE = re.compile(r'^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})(?:\.\d+)?: ?(.*)$')
cle = lambda c: [int(x) for x in re.findall(r'\d+', os.path.basename(c))]   # même tri que 05b/05c
debut_f = coupure - datetime.timedelta(minutes=FENETRE_MIN)
fin_f = coupure + datetime.timedelta(minutes=FENETRE_MIN)
for c in sorted(glob.glob(f'{dossier}/training_log_*.txt'), key=cle):
    with open(c, encoding='utf-8', errors='replace') as fh:
        lignes = fh.read().splitlines()
    datees = []
    for ligne in lignes:
        m = RE_LIGNE.match(ligne)
        if m:
            datees.append((datetime.datetime.fromisoformat(m.group(1)).replace(tzinfo=UTC), m.group(2).strip()))
    print(f'\n{os.path.basename(c)} : {len(lignes)} lignes')
    if not datees:
        continue
    print(f'  première ligne {heure(datees[0][0])} ; dernière ligne {heure(datees[-1][0])} : « {datees[-1][1][:90]} »')
    epoques = [(d, int(e.group(1))) for d, msg in datees if (e := re.fullmatch(r'Epoch (\d+)', msg))]
    if epoques:
        print(f'  époques {epoques[0][1]} à {epoques[-1][1]} ; « Epoch {epoques[0][1]} » {heure(epoques[0][0])} ; '
              f'« Epoch {epoques[-1][1]} » {heure(epoques[-1][0])}')
        # dernière époque TERMINÉE (ligne « Epoch time » écrite après sa ligne « Epoch N »)
        courante, terminee = None, None
        for d, msg in datees:
            if (e := re.fullmatch(r'Epoch (\d+)', msg)):
                courante = int(e.group(1))
            elif msg.startswith('Epoch time') and courante is not None:
                terminee = (courante, d)
        if terminee:
            print(f'  dernière époque terminée : {terminee[0]} ({heure(terminee[1])}) -> dernier checkpoint_latest '
                  f'que cette session a pu écrire : current_epoch = {(terminee[0] + 1) // 10 * 10}')
    autour = [(d, msg) for d, msg in datees if debut_f <= d <= fin_f and (
        msg.startswith(('Epoch', 'Yayy', 'Training done', 'Using splits')) or
        any(mot in msg.lower() for mot in ('checkpoint', 'error', 'erreur', 'traceback')))]
    if autour:
        print(f'  lignes entre {heure(debut_f)} et {heure(fin_f)} :')
        for d, msg in autour:
            print(f'    {heure(d)}  {msg[:100]}')
print('\nNB : nnU-Net n\'écrit AUCUNE ligne quand il enregistre checkpoint_latest.pth (seulement « Yayy! New best » '
      'avant checkpoint_best.pth) : une sauvegarde se déduit de la fin des époques 9, 19, 29...')

# --- 5. Suivi MLflow du pli (lecture seule, sans copie) --------------------------------------------------
print('\n=== 5. Suivi MLflow du pli ===')
base, fichier_run = f'{MLFLOW}/mlflow.db', f'{dossier}/mlflow_run_id.txt'
if os.path.isfile(base) and os.path.isfile(fichier_run):
    with open(fichier_run, encoding='utf-8') as fh:
        run = fh.read().strip()
    con = sqlite3.connect(f'file:{urllib.parse.quote(base)}?mode=ro&immutable=1', uri=True)
    try:
        print(f'Run {run}. Historique de epoques_terminees / epoques_consolidees (une valeur par mise à jour) :')
        for cle_m, valeur, ts in con.execute(
                "SELECT key, value, timestamp FROM metrics WHERE run_uuid = ? AND key IN "
                "('epoques_terminees', 'epoques_consolidees') ORDER BY timestamp, key", (run,)):
            print(f'  {heure(ts / 1000)}  {cle_m} = {valeur:.0f}')
        for cle_t, valeur in con.execute("SELECT key, value FROM tags WHERE run_uuid = ? AND key LIKE 'session_%' "
                                         "ORDER BY key", (run,)):
            print(f'  tag {cle_t} = {valeur[:120]}')
        doubles = con.execute("SELECT MIN(step), MAX(step), COUNT(DISTINCT step) FROM (SELECT step FROM metrics "
                              "WHERE run_uuid = ? AND key = 'train_loss' GROUP BY step HAVING COUNT(*) > 1)",
                              (run,)).fetchone()
        print('Époques notées deux fois dans MLflow (cas de secours remplacer_depuis de 05c) :',
              'aucune' if not doubles[2] else f'{doubles[2]} époques, de {doubles[0]} à {doubles[1]}')
    except sqlite3.Error as err:
        print('Lecture MLflow impossible :', err)
    finally:
        con.close()
else:
    print('Base MLflow ou mlflow_run_id.txt absent.')

# --- 6. Journal du script 05c ------------------------------------------------------------------------------
print('\n=== 6. Journal du script 05c ===')
print('05c n\'écrit pas de fichier journal : sa sortie (« REPRISE depuis ... : époque N », « ATTENTION : reprise à '
      'l\'époque ..., mais les journaux indiquaient ... ») n\'existe que dans la sortie de la cellule de la section 15 '
      'de la session concernée. Les traces écrites par 05c sont dans MLflow (section 5 ci-dessus).')

## 18. Étape 5 — Densité des vaisseaux sur les 15 cas d'entraînement (phase d'injection)
**Avant :** après un redémarrage de session, relance les sections 1 (Drive) et 2 (dépôt à jour). **Pas besoin de GPU.** `nibabel`, `numpy`, `scipy`, `matplotlib` et `pandas` sont déjà dans Colab (si `nibabel` manque, relance la section 3).

**Pourquoi.** La section 16 a montré que l'échec du pli 0 sur `mmwhs_1019` vient probablement de la **phase d'injection** (le moment où les images sont prises après l'injection du produit de contraste) : sur 1019, l'aorte (253 HU) est **moins** dense que l'artère pulmonaire (AP, 416 HU), alors que sur 1001 c'est l'inverse (aorte 562 HU, AP 192 HU) (D27). Deux questions :
1. **1019 est-il le seul** cas d'entraînement dans cette situation, ou y en a-t-il d'autres, éventuellement « limites » ?
2. Quelles densités sont **normales** sur nos 15 cas ? Ces chiffres serviront plus tard à fixer les seuils du **contrôle de plausibilité** (D28, `docs/specification_controle_plausibilite.md`). Ici, **aucun seuil n'est fixé** : on décrit seulement.

**Ce que fait `scripts/05e_densites_vaisseaux.py`**, pour chacun des 15 cas d'entraînement, **un scanner à la fois** en mémoire, d'après la **vérité terrain** (les masques des experts, `labelsTr`) et l'image (`imagesTr`) :
- **densité** en HU (unités Hounsfield : eau = 0, air = −1000, sang injecté environ 250 à 500) de l'**aorte** (étiquette 6), de l'**AP** (7), de la **cavité du ventricule gauche** (VG, 3) et du **ventricule droit** (VD, 5), étiquettes vérifiées dans `dataset.json`. Pour chaque région : la **médiane** (la valeur du milieu, peu sensible aux voxels extrêmes), **p5** et **p95** (5 % des voxels sont en dessous de p5, 5 % au-dessus de p95) ;
- la même médiane au **« cœur »** de la région, c'est-à-dire les voxels à plus de 3 mm de son bord (même définition que la section 16). Au bord, un voxel mélange le sang et la paroi : sa densité est faussée (*effet de volume partiel*). **Cette médiane au cœur est la mesure de référence** pour le contrôle de plausibilité (colonnes `_ref`). Si une région est trop fine pour avoir un cœur, on prend la région entière et on le note (colonne « repli ») ;
- les écarts **aorte − AP**, **aorte − VG**, **AP − VD** et le rapport **aorte / AP**. Le sang de l'aorte sort du VG, celui de l'AP sort du VD : aorte ≈ VG et AP ≈ VD sont attendus quelle que soit la phase ;
- l'**épaisseur de coupe** (famille 0,45 ou 0,625 mm) et le **groupe patient** (paires 1001/1020 et 1005/1012 : même patient probable, D20), pour la lecture ;
- un **drapeau** : **INVERSÉ** si la médiane de l'aorte est plus basse que celle de l'AP (comme 1019) ; **proche** si l'aorte est moins de 20 % plus dense que l'AP (rapport aorte/AP < 1,20). Ce repère de 1,20 est **indicatif**, choisi à l'avance sans validation : il sert à attirer l'œil sur les cas limites, pas à décider. Le drapeau est calculé sur la région entière (colonne `drapeau`) et sur la référence (colonne `drapeau_ref`) ; s'ils diffèrent, le cas est à regarder de près ;
- la **répartition** sur les cas (minimum, médiane, maximum, et quel cas donne le minimum et le maximum), calculée **avec les 15 cas**, **sans 1019** (la spécification exclut 1019 du calibrage de la règle « aorte pas moins dense que l'AP », pour ne pas tourner en rond) et **sans les cas INVERSÉS**.

**Lecture seule, et jamais le lot de test.** Le script lit seulement `imagesTr`, `labelsTr` et `dataset.json` de `Dataset101_MMWHS`. Il **refuse** tout cas qui n'est pas en entraînement dans le découpage figé `splits/mmwhs_split.json` (lu, jamais modifié) : les cas de test 1004, 1010, 1011, 1013 et 1014 sont rejetés (code 1). Il n'écrit **que** dans `poc_ct/analyses/05e_densites/` : `densites_vaisseaux.csv` (une ligne par cas), `synthese_05e.json` (lignes, répartitions, définitions, anomalies) et `densites_vaisseaux.png` (figure). Il s'arrête si ce dossier est placé dans un dossier nnU-Net ou dans le dépôt.

**Anomalies signalées, jamais corrigées** (lignes « ANOMALIE ») : en-tête du masque différent de celui de l'image, étiquette inattendue, structure absente de la vérité, épaisseur de coupe hors des deux familles. Un cas dont le masque n'a pas la même taille que l'image n'est pas mesuré (drapeau « incomplet »).

**Lire la figure.** En haut, pour chaque cas, la médiane (point) et l'étendue p5-p95 (barre) de l'aorte (rond bleu), de l'AP (carré orange), du VG (triangle vert) et du VD (losange jaune), sur la région entière. Les cas sont rangés du rapport aorte/AP le plus bas au plus haut : **les cas suspects sont à gauche**. Fond rouge et mot « INVERSÉ » = aorte moins dense que l'AP ; fond jaune et mot « proche » = repère indicatif. En bas, le rapport aorte/AP, avec la ligne 1 (aorte = AP) et la ligne pointillée 1,20. Sous chaque cas : son épaisseur de coupe et, s'il y a lieu, sa paire de patient.

**Points d'attention.**
- Les deux familles d'épaisseur (0,45 et 0,625 mm, probablement deux scanners) peuvent avoir des densités différentes : regarder si les cas « proches » appartiennent tous à la même famille.
- Les deux cas d'une même paire (1001/1020, 1005/1012) comptent pour **un seul patient** : deux valeurs voisines ne sont pas deux confirmations indépendantes.
- 15 cas, dont 13 patients : les minimum et maximum sont fragiles. Un seul cas de plus peut les déplacer.

**Durée** : environ 5 à 10 minutes (15 scanners lus depuis Drive, un à la fois).

**Résultat attendu :** « 15 cas vérifiés, tous en ENTRAÎNEMENT », une ligne de lecture par cas, le tableau trié, la répartition « tous » / « sans_1019 », d'éventuelles lignes « ANOMALIE », puis « Écrit dans ... ». Code d'arrêt 1 = garde-fou (cas refusé, fichier introuvable, dossier de sortie interdit, étiquettes inattendues dans `dataset.json`) : rien n'est écrit.

Envoie **toute** la sortie de la cellule dans la conversation, ainsi que le tableau et la figure affichés par la cellule suivante.

In [ ]:
RAW = '/content/drive/MyDrive/poc_ct/nnunet_raw'                    # seuls imagesTr, labelsTr et dataset.json sont lus
SORTIE = '/content/drive/MyDrive/poc_ct/analyses/05e_densites'      # HORS des dossiers nnU-Net : CSV + JSON + figure
# LECTURE SEULE. Sans --cas : les 15 cas d'entraînement du découpage figé, un scanner à la fois en mémoire.
# Tout cas hors du lot d'entraînement est refusé (test : 1004, 1010, 1011, 1013, 1014).
!python scripts/05e_densites_vaisseaux.py --raw "$RAW" --sortie "$SORTIE"

In [ ]:
# Affiche le tableau trié, la répartition et la figure écrits par le script dans Drive. LECTURE SEULE.
import json, os
import pandas as pd
from IPython.display import Image, display

SORTIE = '/content/drive/MyDrive/poc_ct/analyses/05e_densites'
tableau, synthese, figure = (f'{SORTIE}/{n}' for n in ('densites_vaisseaux.csv', 'synthese_05e.json', 'densites_vaisseaux.png'))
if not all(os.path.exists(x) for x in (tableau, synthese, figure)):
    print('Fichiers introuvables dans', SORTIE, ": lancer d'abord la cellule précédente")
else:
    # Tableau déjà trié par le script (rapport aorte/AP croissant : cas suspects en haut). « ref » = médiane au cœur.
    texte = ['groupe_patient', 'drapeau', 'drapeau_ref', 'ref_repli_region_entiere']   # colonnes de texte
    df = pd.read_csv(tableau, dtype={c: str for c in texte}).convert_dtypes()   # <NA> = valeur absente
    df[texte] = df[texte].fillna('')
    colonnes = {'cas': 'cas', 'epaisseur_coupe_mm': 'épaisseur (mm)', 'groupe_patient': 'paire patient',
                'aorte_mediane': 'aorte', 'AP_mediane': 'AP', 'VG_mediane': 'VG', 'VD_mediane': 'VD',
                'aorte_ref': 'aorte réf', 'AP_ref': 'AP réf', 'VG_ref': 'VG réf', 'VD_ref': 'VD réf',
                'aorte_moins_AP_ref': 'aorte−AP réf', 'aorte_moins_VG_ref': 'aorte−VG réf', 'AP_moins_VD_ref': 'AP−VD réf',
                'aorte_sur_AP': 'aorte/AP', 'aorte_sur_AP_ref': 'aorte/AP réf',
                'drapeau': 'drapeau', 'drapeau_ref': 'drapeau réf', 'ref_repli_region_entiere': 'repli (sans cœur)'}
    print('Densités médianes en HU, vérité terrain. Sans suffixe : région entière ; « réf » : cœur de la région (> 3 mm du bord).')
    display(df[list(colonnes)].rename(columns=colonnes))

    with open(synthese, encoding='utf-8') as f:
        s = json.load(f)
    for a in s['avertissements']:
        print('AVERTISSEMENT :', a)
    for qui, liste in s['anomalies'].items():
        for a in liste:
            print(f'ANOMALIE {qui} : {a}')
    print('Cas signalés (drapeau sur la région entière) :', s['cas_signales'] or 'aucun')
    # Répartition sur les cas (mesure de référence) : avec les 15 cas, sans 1019, sans les cas INVERSÉS.
    lignes = [{'grandeur': cle.removesuffix('_ref'), 'cas retenus': g, **{k: r[k] for k in ('n', 'min', 'cas_min', 'mediane', 'max', 'cas_max')}}
              for g, d in s['repartition'].items() for cle, r in d.items() if cle.endswith('_ref') and r]
    display(pd.DataFrame(lignes))
    display(Image(filename=figure))

## 19. Étape 5 — Bilan de la validation croisée : les 15 cas dans un seul tableau
**Quand :** à la fin des 5 plis (section 15). Avant, le script marche aussi, mais le bilan est **partiel** (code de sortie 2) : utile pour suivre, pas pour conclure.
**Avant :** après un redémarrage de session, relance les sections 1 (Drive) et 2 (dépôt à jour). **Pas besoin de GPU** ; `nibabel` est déjà dans Colab (sinon, section 3).

**Pourquoi.** Chaque pli se termine par une **validation finale** : son modèle prédit ses 3 cas de validation, qu'il n'a jamais vus, et nnU-Net écrit les scores dans `fold_X/validation/summary.json`. Chaque cas d'entraînement est en validation dans **un seul** pli : les 5 plis réunis donnent un score **« hors pli »** pour chacun des 15 cas. C'est notre meilleure estimation de la qualité du modèle **avant** le test, qui reste sous clé jusqu'à l'étape 6.

**Ce que fait `scripts/05f_bilan_validation_croisee.py`** :
- vérifie `splits_final.json` comme les scripts 05b et 05c, puis lit **seulement** les 5 `summary.json` (aucune image) ; arrêt si un fichier ne contient pas exactement les cas de validation de son pli (code 6) ;
- recalcule le Dice moyen de chaque pli et le compare à celui écrit par nnU-Net (contrôle de lecture) ;
- affiche le **tableau des 15 cas** : Dice par structure (1 myocarde VG, 2 OG, 3 cavité VG, 4 OD, 5 VD, 6 aorte, 7 AP), moyenne par cas, pli, épaisseur de coupe et **patient** ;
- par structure : **moyenne sur les cas** et **moyenne par patient**, médiane, écart-type (dispersion), minimum (avec le cas), maximum, moyenne par épaisseur de coupe ;
- **signale** les cellules faibles (Dice < 0,85, marquées `*`) et les échecs (Dice < 0,50, marqués `!`), avec la part d'**oublis** (vérité non prédite) et d'**ajouts** (prédit en trop). Ces repères sont **indicatifs**, fixés avant de voir les 5 plis ; ils ne décident de rien ;
- liste les 3 cas à la moyenne la plus basse et **propose la commande** de la section 20 pour chaque cas signalé.

**Rappel : 1001/1020 et 1005/1012 sont deux examens du même patient probable (D20).** Ils sont dans le même pli, donc jamais vus par le modèle qui les prédit, mais ils ne sont **pas indépendants** : deux bons scores sur une paire valent une seule confirmation. D'où la **moyenne par patient** (13 patients pour 15 cas), à regarder à côté de la moyenne sur les cas.

**Lecture seule.** Écrit **uniquement** dans `poc_ct/analyses/05f_bilan/` : `bilan_validation_croisee.csv` (une ligne par cas) et `synthese_05f.json`. Aucun cas de test n'est lu.

**Durée** : moins d'une minute.

**Résultat attendu :** une ligne par pli (« terminé » avec les deux moyennes identiques), « BILAN COMPLET : 15 cas de validation, 5 plis », le tableau, le rappel des paires, les statistiques, les cellules signalées, les commandes pour la section 20, puis « Écrit dans ... ». Codes d'arrêt : 1 = garde-fou ; 2 = bilan partiel (plis non terminés) ; 6 = `summary.json` incohérent (ne pas utiliser ces résultats).

Envoie **toute** la sortie de la cellule dans la conversation, ainsi que le tableau coloré affiché par la cellule suivante.

In [ ]:
RAW = '/content/drive/MyDrive/poc_ct/nnunet_raw'                    # seuls les en-têtes de imagesTr sont lus
PRE = '/content/drive/MyDrive/poc_ct/nnunet_preprocessed'           # splits_final.json et fiches de 05a (lus)
RES = '/content/drive/MyDrive/poc_ct/nnunet_results'                # seuls les fold_X/validation/summary.json sont lus
SORTIE = '/content/drive/MyDrive/poc_ct/analyses/05f_bilan'         # HORS des dossiers nnU-Net : CSV + synthèse
# LECTURE SEULE. Code de sortie 2 = bilan partiel (plis pas tous terminés) : normal avant la fin des 5 plis.
!python scripts/05f_bilan_validation_croisee.py --raw "$RAW" --preprocessed "$PRE" --results "$RES" --sortie "$SORTIE"

In [ ]:
# Affiche le tableau des 15 cas, coloré (rouge = Dice bas). LECTURE SEULE : rien n'est recalculé.
import os
import pandas as pd
from IPython.display import display

SORTIE = '/content/drive/MyDrive/poc_ct/analyses/05f_bilan'
chemin = f'{SORTIE}/bilan_validation_croisee.csv'
if os.path.exists(chemin):
    t = pd.read_csv(chemin, dtype={'cas': str, 'patient': str})
    colonnes_dice = [c for c in t.columns if c.startswith('dice_')] + ['moyenne_cas']
    display(t.style.format('{:.3f}', subset=colonnes_dice, na_rep='—')
             .background_gradient(cmap='RdYlGn', subset=colonnes_dice, vmin=0.5, vmax=1.0))
else:
    print(f'Tableau introuvable : {chemin} (lancer d\'abord la cellule précédente)')

## 20. Étape 5 — Examiner un cas de validation faible (05d généralisé)
**Quand :** après la section 19, pour les cas qu'elle signale. Premier cas prévu : **1015** (pli 1 : VD 0,83 et AP 0,65, avec à la fois des oublis et des ajouts ; phase d'injection classique).
**Avant :** après un redémarrage de session, relance les sections 1 (Drive) et 2 (dépôt à jour). **Pas besoin de GPU.**

**Pourquoi.** Le script 05d (section 16) a été écrit pour un seul échec : les vaisseaux de 1019. `scripts/05g_examiner_cas.py` fait **le même examen, avec les mêmes figures**, pour **n'importe quel cas de validation** des 5 plis et **n'importe quelles structures**. Il retrouve tout seul le pli de chaque cas. Il aide à trancher entre quatre hypothèses :
- **(a) frontière** : la limite entre deux structures voisines est placée ailleurs par le modèle que par l'annotateur. Exemple : la limite VD / AP, au niveau de la **valve pulmonaire** (la valve qui sépare le ventricule droit de l'artère pulmonaire). Signe distinctif : les erreurs sont **croisées** (les oublis du VD sont prédits « AP » et les ajouts de l'AP sont posés sur le VD de la vérité, ou l'inverse). Petit décalage : un **liseré** (bande fine) le long du bord ; grand décalage : des erreurs loin et au-delà de la boîte, mais toujours croisées ;
- **(b) étendue** : l'annotation ou la prédiction va **plus loin**, par exemple dans les **branches** de l'artère pulmonaire (elle se divise en branche droite et branche gauche, et l'annotateur peut s'arrêter plus ou moins loin). Les erreurs sont alors des **morceaux entiers**, loin de l'autre masque, au-delà de sa boîte englobante (le plus petit pavé qui le contient) ;
- **(c) confusion ou oubli franc** : un morceau est pris pour une autre structure ou pour le fond ;
- **(d) examen inhabituel** : phase d'injection, bruit, contraste, cadrage (comme 1019, D27).

**Ce que fait le script**, pour chaque cas (le 1er = cas étudié, les suivants = comparaison) :
1. **géométrie** et **Dice des 7 structures**, recalculé et comparé au `summary.json` du pli (un écart est signalé « ANOMALIE ») ;
2. pour chaque structure examinée, vérité **et** prédiction : volume, niveaux haut et bas, boîte englobante, distance aux bords du champ, centre de gravité, morceaux (comme 05d) ;
3. les **erreurs** : **oublis** (vérité non prédite) et **ajouts** (prédit hors de la vérité). Pour chacun : ce que l'autre masque y met (fond ? structure voisine ?), la **distance à la même structure dans l'autre masque** (la prédiction pour un oubli, la vérité pour un ajout ; **près** = 2 mm ou moins, **loin** = plus de 5 mm, le reste est entre les deux), la part **au-delà de la boîte** de l'autre masque (au-dessus, en dessous, sur les côtés), les niveaux, les morceaux et la densité ;
4. la **matrice de confusion** complète, les **densités** des 7 structures, la **phase d'injection** (rapport aorte / AP, même repère que la section 18), le bruit et le CNR ;
5. une **figure** par cas, lue comme celle de la section 16 : une rangée par structure examinée ; colonnes axiale, coronale, sagittale, projection de face. Couleurs dans l'ordre des structures : 1re structure **rouge** (vérité) / **jaune** (prédite), 2e **bleu** / **magenta**, 3e vert / cyan, 4e orange / blanc ; trait épais = vérité, trait fin = prédiction ;
6. un **récapitulatif** rangé par hypothèse, cas côte à côte. **Le script ne conclut pas.**

**Choix des structures** : `--structures 5 7` (de 1 à 4 étiquettes). Sans cette option : les 2 structures au Dice le plus bas du cas étudié.
**Choix du cas de comparaison** : un cas réussi **de la même épaisseur de coupe**, si possible d'un autre patient ; la section 19 en propose un. Pour 1015, 1005 est dans le même pli (même modèle), mais 1005 et 1012 sont le même patient : n'en prendre qu'un.

**Lecture seule, et jamais le lot de test.** Lit seulement `imagesTr`, `labelsTr` et `dataset.json`, et `fold_X/validation/` du pli de chaque cas. Refuse tout cas qui n'est pas un cas de validation (les cas de test 1004, 1010, 1011, 1013 et 1014 sont rejetés, code 1). Écrit **uniquement** dans `poc_ct/analyses/05g_<cas>/` (figures `examen_mmwhs_XXXX.png` et `synthese_05g.json`).

**Durée** : quelques minutes (un scanner à la fois).

**Résultat attendu :** « Cas vérifiés : ... », les tableaux 1 à 6 pour chaque cas et « Figure : ... », le champ de vue des 15 cas d'entraînement, « 8. RÉCAPITULATIF » et « Synthèse : ... ». Codes d'arrêt : 1 = garde-fou (rien n'est écrit) ; 6 = image, vérité et prédiction de tailles différentes.

Envoie **toute** la sortie de la cellule dans la conversation, ainsi que les figures et le tableau affichés par la cellule suivante.

In [ ]:
CAS = '1015 1005'                                                   # 1er = cas étudié ; suivants = comparaison
STRUCTURES = '5 7'                                                  # 5 = VD, 7 = AP ; laisser '' pour les 2 plus faibles
RAW = '/content/drive/MyDrive/poc_ct/nnunet_raw'                    # seuls imagesTr, labelsTr et dataset.json sont lus
PRE = '/content/drive/MyDrive/poc_ct/nnunet_preprocessed'           # splits_final.json (pli de chaque cas), lu seulement
RES = '/content/drive/MyDrive/poc_ct/nnunet_results'                # seul fold_X/validation est lu
SORTIE = f"/content/drive/MyDrive/poc_ct/analyses/05g_{CAS.split()[0]}"   # HORS des dossiers nnU-Net
OPTION = f'--structures {STRUCTURES}' if STRUCTURES.strip() else ''
# LECTURE SEULE. Tout cas qui n'est pas un cas de validation est refusé (test : 1004, 1010, 1011, 1013, 1014).
!python scripts/05g_examiner_cas.py --cas $CAS $OPTION --raw "$RAW" --preprocessed "$PRE" --results "$RES" --sortie "$SORTIE"

In [ ]:
# Affiche les figures et le récapitulatif écrits par le script dans Drive. LECTURE SEULE : rien n'est recalculé.
import json, os
import pandas as pd
from IPython.display import Image, display

CAS = '1015 1005'                                                   # mêmes cas que la cellule précédente
SORTIE = f"/content/drive/MyDrive/poc_ct/analyses/05g_{CAS.split()[0]}"
synthese = f'{SORTIE}/synthese_05g.json'
if not os.path.exists(synthese):
    print(f'Synthèse introuvable : {synthese} (lancer d\'abord la cellule précédente)')
else:
    with open(synthese, encoding='utf-8') as f:
        s = json.load(f)
    for cas, figure in s['figures'].items():
        print(f'{cas} — trait épais = vérité, trait fin = prédiction (légende sous la figure)')
        display(Image(filename=figure))
    # Récapitulatif : une ligne par indicateur, une colonne par cas.
    lignes = []
    for cle_structure in next(iter(s['indicateurs'].values())):
        lecture = s['lecture']['examen'] if cle_structure == 'examen' else s['lecture']['structure']
        for cle, l in lecture.items():
            lignes.append({'structure': cle_structure, 'hypothèse': l.get('hypothese', '(d)'), 'indicateur': l['libelle'],
                           **{cas[-4:]: ind[cle_structure][cle] for cas, ind in s['indicateurs'].items()},
                           'si la valeur est inhabituelle': l['si']})
    pd.set_option('display.max_colwidth', None)
    display(pd.DataFrame(lignes))